# Advanced Python for Generative AI - Prep Work Training (all modules)

Run cells top to bottom with Shift + Enter. Modules 1-5 cells are independent; Module 6 cells must run in order.

# Module 1: Advanced Python for GenAI (30 min)

The pre-program covered Python basics: variables, data types, operators, control flow, lists, tuples and dictionaries. This module adds the Python features you will meet in every Generative AI code base: comprehensions, flexible functions, decorators, generators, classes, JSON handling with error control, and NumPy/Pandas.

![Figure 1: The learning path in this training. Each module is a building block for GenAI.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d01_roadmap.png)
*Figure 1: The learning path in this training. Each module is a building block for GenAI.*


![Figure 2: Where each Python feature in this module is used inside a typical GenAI (RAG) application.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d02_python_genai.png)
*Figure 2: Where each Python feature in this module is used inside a typical GenAI (RAG) application.*


> **How to run the code**
>
> Open colab.research.google.com, click New notebook, paste each code block into its own cell and press Shift + Enter. Every block in Modules 1 to 5 is self-contained, so it can be run on its own and in any order. All libraries used (NumPy, Pandas, SciPy, Matplotlib, Seaborn, scikit-learn, TensorFlow/Keras) are pre-installed in Colab.

## 1.1 Comprehensions

A comprehension builds a new list, dictionary or set from an existing collection in a single readable line. The pattern is `[expression for item in collection if condition]`. It replaces the "create empty list, loop, append" pattern and is usually faster because the loop runs inside Python's C implementation.

- **List comprehension** `[...]`: transform and/or filter items.
- **Dict comprehension** `{k: v ...}`: build key-value lookups, e.g. product to price.
- **Set comprehension** `{...}`: unique values only.
- **Nested comprehension**: flatten lists of lists, a very common step when working with tokenised text.

> **Real-world example**
>
> An e-commerce team scrapes product titles from partner sites. The titles arrive with extra spaces, random capitalisation, blanks and missing values. Before they are sent to an embedding model they must be cleaned and de-duplicated. One comprehension does the job.

In [ ]:
# Real-world example: cleaning raw product titles scraped from an e-commerce site
raw_titles = ["  Wireless EARBUDS ", "smart watch", "", "  Yoga Mat", None, "AIR fryer  "]

# 1) List comprehension with a filter: keep valid titles and clean them
clean_titles = [t.strip().title() for t in raw_titles if t and t.strip()]
print("Clean titles:", clean_titles)

# 2) Dict comprehension: product -> price after a 10% festival discount
prices = {"Earbuds": 1999, "Smart Watch": 3999, "Yoga Mat": 899}
sale_prices = {name: round(price * 0.9, 2) for name, price in prices.items()}
print("Sale prices:", sale_prices)

# 3) Set comprehension: unique categories (duplicates removed automatically)
orders = [("Earbuds", "Electronics"), ("Yoga Mat", "Sports"), ("Watch", "Electronics")]
categories = {category for _, category in orders}
print("Unique categories:", sorted(categories))

# 4) Conditional expression inside a comprehension: label each order
order_values = [450, 2300, 999, 5600]
labels = ["High" if v > 2000 else "Low" for v in order_values]
print("Order labels:", labels)

# 5) Nested comprehension: flatten tokenised sentences (very common in NLP)
sentences = [["deliver", "fast"], ["great", "quality"], ["return", "item"]]
tokens = [word for sentence in sentences for word in sentence]
print("Flattened tokens:", tokens)

## 1.2 Functions: default arguments, *args, **kwargs and lambda

Functions package logic so it can be reused and tested. Four features make functions flexible:

- **Default arguments** (`role="assistant"`) let callers skip common options.
- **\*args** collects any number of positional values into a tuple, e.g. any number of item prices.
- **\*\*kwargs** collects any number of named options into a dictionary. LLM SDKs use this pattern heavily, for example `client.chat(model=..., temperature=..., max_tokens=...)`.
- **lambda** creates a small one-line function, usually passed to `sorted`, `filter` or `map`.

> **Real-world example**
>
> Every GenAI application builds prompts. A prompt builder function takes the user's question, optional context retrieved from documents, and any number of extra instructions (tone, length, language) and returns one prompt string. The same function serves the chatbot, the email writer and the product-description generator.

In [ ]:
# Real-world example: a reusable prompt builder for an LLM application
def build_prompt(question, context="", role="helpful shopping assistant", **options):
    """Return a prompt string. Extra keyword options become instructions."""
    prompt = f"You are a {role}.\n"
    if context:
        prompt += f"Use only this context:\n{context}\n"
    prompt += f"Question: {question}\n"
    for key, value in options.items():
        prompt += f"Instruction - {key}: {value}\n"
    return prompt

print(build_prompt("What is the return window?",
                   context="Products can be returned within 7 days of delivery.",
                   tone="friendly", max_words=40))

# *args collects any number of positional values into a tuple
def total_bill(*item_prices, gst_rate=0.18):
    subtotal = sum(item_prices)
    return round(subtotal * (1 + gst_rate), 2)

print("Bill with GST:", total_bill(499, 1299, 250))

# lambda + sorted / filter / map on a product list
products = [
    {"name": "Earbuds", "price": 1999, "rating": 4.3},
    {"name": "Yoga Mat", "price": 899, "rating": 4.6},
    {"name": "Air Fryer", "price": 5999, "rating": 4.1},
    {"name": "Green Tea", "price": 349, "rating": 4.4},
]
top_rated = sorted(products, key=lambda p: p["rating"], reverse=True)
under_1000 = list(filter(lambda p: p["price"] < 1000, products))
names_upper = list(map(lambda p: p["name"].upper(), products))

print("Top rated:", [p["name"] for p in top_rated])
print("Under Rs 1000:", [p["name"] for p in under_1000])
print("Upper-case names:", names_upper)

## 1.3 Decorators

A decorator is a function that wraps another function to add behaviour (logging, timing, caching, retrying) without changing the original code. Writing `@timer` above a function is shorthand for `func = timer(func)`. A decorator factory such as `@retry(max_attempts=5)` is a function that returns a decorator, so it can take its own settings. `functools.wraps` keeps the original function's name and docstring.

> **Real-world example**
>
> LLM APIs fail now and then with timeouts or rate-limit errors. Production code wraps every API call with a retry decorator (often with a growing wait time) and a timer to monitor latency. Libraries such as `tenacity` and LangChain's retry logic work exactly this way.

In [ ]:
import time
import random
import functools

def timer(func):
    """Decorator: prints how long a function took."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"[timer] {func.__name__} took {time.perf_counter() - start:.3f} s")
        return result
    return wrapper

def retry(max_attempts=3, wait_seconds=0.2):
    """Decorator factory: retries a function when it raises ConnectionError."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as err:
                    print(f"[retry] attempt {attempt} failed: {err}")
                    time.sleep(wait_seconds)
            raise RuntimeError(f"{func.__name__} failed after {max_attempts} attempts")
        return wrapper
    return decorator

random.seed(7)

@timer
@retry(max_attempts=5)
def call_llm_api(prompt):
    """Simulates an unreliable LLM API that fails about half the time."""
    if random.random() < 0.5:
        raise ConnectionError("API timeout")
    return f"LLM answer to: '{prompt}'"

print(call_llm_api("Summarise yesterday's sales in one line"))

## 1.4 Generators

A generator is a function that uses `yield` instead of `return`. It produces values one at a time, only when asked, and remembers where it stopped. This is called lazy evaluation. A list of one million numbers uses about 8 MB; a generator for the same numbers uses about 200 bytes.

- Use generators for large files, streaming API responses and batching.
- A generator expression `(x for x in data)` is the lazy version of a list comprehension.
- When an LLM "streams" its answer word by word, the client library is iterating over a generator.

> **Real-world example**
>
> Retrieval-Augmented Generation (RAG) starts by splitting long documents (return policy, product manuals) into overlapping chunks, because embedding models accept limited text. The overlap keeps sentences that cross a boundary from losing meaning. Chunks are then sent to the embedding API in batches.

In [ ]:
import sys

# Real-world example: splitting a long document into overlapping chunks before
# sending it to an embedding model (this is step 1 of every RAG pipeline)
def chunk_text(text, chunk_size=10, overlap=3):
    """Yield chunks of `chunk_size` words that overlap by `overlap` words."""
    words = text.split()
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        yield " ".join(words[start:start + chunk_size])
        if start + chunk_size >= len(words):
            break

policy = ("Customers can return most products within 7 days of delivery. "
          "Electronics must be unused and in the original packaging. "
          "Refunds are processed to the original payment method within 5 business days. "
          "Cash on delivery orders are refunded to the bank account given by the customer.")

for i, chunk in enumerate(chunk_text(policy), start=1):
    print(f"Chunk {i}: {chunk}")

# Batching: send reviews to an API in groups instead of one huge request
def batch(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]

reviews = [f"review_{i}" for i in range(1, 11)]
for b in batch(reviews, 4):
    print("Sending batch:", b)

# Why generators? They produce values lazily and use almost no memory
squares_list = [x * x for x in range(1_000_000)]
squares_gen = (x * x for x in range(1_000_000))
print(f"List size: {sys.getsizeof(squares_list):,} bytes")
print(f"Generator size: {sys.getsizeof(squares_gen):,} bytes")

## 1.5 Classes and dataclasses

Object-oriented programming groups data (attributes) and behaviour (methods) into a class. A `@dataclass` automatically writes the `__init__`, `__repr__` and comparison methods for classes that mainly hold data, so the code stays short. `field(default_factory=list)` gives every object its own empty list.

> **Real-world example**
>
> A chatbot must send the conversation history to the LLM with every request, but models have a context limit (measured in tokens; we use words here for simplicity). A `Conversation` class stores messages, keeps the system instruction, and trims the oldest turns when the budget is exceeded. It also converts messages to the `[{"role": ..., "content": ...}]` format that LLM APIs expect.

In [ ]:
from dataclasses import dataclass, field
from typing import List

# Real-world example: managing chat history for an LLM chatbot with a word budget
@dataclass
class Message:
    role: str        # "system", "user" or "assistant"
    content: str

    def word_count(self):
        return len(self.content.split())

@dataclass
class Conversation:
    max_words: int = 30
    messages: List[Message] = field(default_factory=list)

    def add(self, role, content):
        self.messages.append(Message(role, content))
        self._trim()

    def total_words(self):
        return sum(m.word_count() for m in self.messages)

    def _trim(self):
        # Keep the system message (index 0); drop the oldest turns when over budget
        while self.total_words() > self.max_words and len(self.messages) > 2:
            removed = self.messages.pop(1)
            print(f"  (trimmed old message: '{removed.content[:30]}...')")

    def to_api_format(self):
        return [{"role": m.role, "content": m.content} for m in self.messages]

chat = Conversation(max_words=30)
chat.add("system", "You are a helpful e-commerce support assistant.")
chat.add("user", "Where is my order ORD200145? It was due yesterday.")
chat.add("assistant", "Your order is out for delivery and will arrive today by 7 PM.")
chat.add("user", "Can I change the delivery address to my office in Koramangala?")

print("Words in history:", chat.total_words())
for msg in chat.to_api_format():
    print(msg)

## 1.6 Inheritance and polymorphism

Inheritance lets a child class reuse and extend a parent class. Polymorphism means different classes respond to the same method call in their own way. A base class defines the interface (`generate()`), and each child provides its own implementation. Code that uses the models does not care which model it receives.

> **Real-world example**
>
> Frameworks such as LangChain and LlamaIndex define a base LLM class; OpenAI, Anthropic, Gemini and local models are child classes. You can switch providers by changing one line. Teams also write cheap fake models (like `EchoLLM` below) to test their pipelines without paying for API calls.

In [ ]:
# Real-world example: the "common interface, many models" pattern used by
# LangChain / LlamaIndex. Every model class exposes the same generate() method.
class BaseLLM:
    def __init__(self, name, temperature=0.0):
        self.name = name
        self.temperature = temperature

    def generate(self, prompt):
        raise NotImplementedError("Child classes must implement generate()")

    def __repr__(self):
        return f"{self.__class__.__name__}(name='{self.name}', temperature={self.temperature})"

class EchoLLM(BaseLLM):
    """A fake model for testing pipelines without paying for API calls."""
    def generate(self, prompt):
        return f"[{self.name}] You said: {prompt}"

class FAQBot(BaseLLM):
    """A rule-based 'model' that answers from a small knowledge base."""
    def __init__(self, name, faq):
        super().__init__(name)
        self.faq = faq

    def generate(self, prompt):
        for keyword, answer in self.faq.items():
            if keyword in prompt.lower():
                return f"[{self.name}] {answer}"
        return f"[{self.name}] Sorry, I will connect you to a human agent."

faq = {"refund": "Refunds reach your account in 5 business days.",
       "cancel": "You can cancel before the order is shipped."}

models = [EchoLLM("echo-v1"), FAQBot("faq-bot", faq)]
for model in models:                       # polymorphism: same call, different behaviour
    print(model)
    print("  ->", model.generate("How long does a refund take?"))

## 1.7 JSON and exception handling

JSON is the standard format for data exchanged with APIs. `json.loads` converts a JSON string to a Python dictionary and `json.dumps`/`json.dump` does the reverse. Exception handling keeps a program running when something goes wrong:

- `try`: code that might fail.
- `except`: what to do if a specific error happens.
- `else`: runs only if no error happened.
- `finally`: always runs (closing files, releasing connections).

> **Real-world example**
>
> We ask an LLM to return sentiment as JSON. Sometimes it returns clean JSON, sometimes it adds friendly text around the JSON, sometimes it returns plain text or broken JSON. A robust parser tries strict parsing first, then extracts the `{...}` block with a regular expression, and finally falls back to a safe default instead of crashing the pipeline.

In [ ]:
import json
import re

# Real-world example: LLMs are asked to return JSON, but they do not always obey.
llm_outputs = [
    '{"sentiment": "positive", "score": 0.92}',
    'Sure! Here is the result:\n{"sentiment": "negative", "score": 0.15}\nHope this helps.',
    'I think the sentiment is neutral.',
    '{"sentiment": "positive", "score": 0.8',          # broken JSON (missing brace)
]

def parse_llm_json(text):
    """Try strict JSON first, then extract a {...} block, else return a safe default."""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass
    return {"sentiment": "unknown", "score": None}

results = []
for raw in llm_outputs:
    parsed = parse_llm_json(raw)
    results.append(parsed)
    print(f"{raw[:35]!r:40} -> {parsed}")

# try / except / else / finally
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero - returning 0")
        result = 0
    else:
        print("Division successful")
    finally:
        print("Done")
    return result

print("Conversion rate:", safe_divide(45, 900))
print("Conversion rate:", safe_divide(45, 0))

# Save and reload results as a JSON file
with open("llm_results.json", "w") as f:
    json.dump(results, f, indent=2)
with open("llm_results.json") as f:
    print("Reloaded from file:", json.load(f))

## 1.8 NumPy and Pandas essentials

**NumPy** stores numbers in fixed-type arrays and applies operations to the whole array at once (vectorisation). This is 50 to 100 times faster than a Python loop and is the foundation of every ML and DL library. **Pandas** adds labelled tables (DataFrames) with SQL-like operations: filter, group by, aggregate, pivot and join.

| SQL | Pandas |
|---|---|
| SELECT col1, col2 | df[["col1", "col2"]] |
| WHERE revenue > 8000 | df[df["revenue"] > 8000] |
| GROUP BY category + SUM | df.groupby("category")["revenue"].sum() |
| JOIN targets ON category | df.merge(targets, on="category") |
| PIVOT | df.pivot_table(index=..., columns=..., values=...) |


> **Real-world example**
>
> A category manager wants revenue by category and city, compared with monthly targets. In Pandas this is a group-by, a pivot table and a merge, the same steps as in SQL.

In [ ]:
import time
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# 1) Vectorisation: NumPy is much faster than a Python loop
prices = np.random.default_rng(0).uniform(100, 5000, 1_000_000)

start = time.perf_counter()
loop_result = [p * 0.9 for p in prices]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = prices * 0.9
vector_time = time.perf_counter() - start
print(f"Loop: {loop_time:.3f} s | NumPy: {vector_time:.4f} s | "
      f"NumPy is ~{loop_time / vector_time:.0f}x faster")

# 2) Pandas: a small order table
rng = np.random.default_rng(42)
n = 1000
orders = pd.DataFrame({
    "order_id": [f"ORD{i}" for i in range(1, n + 1)],
    "category": rng.choice(["Electronics", "Fashion", "Books", "Grocery"], n),
    "city": rng.choice(["Bengaluru", "Mumbai", "Delhi"], n),
    "price": rng.integers(200, 5000, n),
    "quantity": rng.integers(1, 4, n),
})
orders["revenue"] = orders["price"] * orders["quantity"]       # vectorised new column
print(orders.head())

# Filter, group, aggregate
high_value = orders[orders["revenue"] > 8000]
print("\nHigh-value orders:", len(high_value))

summary = (orders.groupby("category")
                 .agg(orders=("order_id", "count"),
                      total_revenue=("revenue", "sum"),
                      avg_revenue=("revenue", "mean"))
                 .sort_values("total_revenue", ascending=False)
                 .round(0))
print("\nRevenue by category:\n", summary)

# Pivot table: city x category revenue
pivot = orders.pivot_table(index="city", columns="category", values="revenue", aggfunc="sum")
print("\nCity x Category revenue:\n", pivot)

# Merge with a targets table (like a SQL JOIN)
targets = pd.DataFrame({"category": ["Electronics", "Fashion", "Books", "Grocery"],
                        "target": [2_000_000, 1_500_000, 900_000, 800_000]})
result = summary.reset_index().merge(targets, on="category")
result["achieved_pct"] = (result["total_revenue"] / result["target"] * 100).round(1)
print("\nTarget achievement:\n", result[["category", "total_revenue", "target", "achieved_pct"]])

## 1.9 Module 1 practical: review-processing pipeline

This practical combines everything from Module 1. Raw customer reviews are cleaned with regular expressions, scored with a simple keyword sentiment function, streamed through a generator, stored as dataclass objects, summarised with Pandas and exported to JSON. In a real GenAI system, the exported negative reviews would be sent to an LLM to summarise root causes.

In [ ]:
import json
import re
from dataclasses import dataclass, asdict
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# PRACTICAL: a mini review-processing pipeline (the kind of pre-processing done
# before reviews are summarised by an LLM)
raw_reviews = [
    ("ORD1", "Electronics", "  GREAT earbuds!!! Sound quality is excellent  "),
    ("ORD2", "Fashion", "Size did not fit. Very disappointed, returning it."),
    ("ORD3", "Fashion", "Nice fabric and fast delivery :)"),
    ("ORD4", "Electronics", "Stopped working after 2 days. Poor quality."),
    ("ORD5", "Books", ""),
    ("ORD6", "Grocery", "Fresh and well packed, good value."),
    ("ORD7", "Electronics", "Okay product, average battery life."),
]

POSITIVE = {"great", "excellent", "nice", "fast", "fresh", "good", "love"}
NEGATIVE = {"poor", "disappointed", "returning", "stopped", "bad", "damaged", "late"}

@dataclass
class Review:
    order_id: str
    category: str
    text: str
    sentiment: str = "neutral"
    score: int = 0

def clean(text):
    text = text.lower().strip()
    return re.sub(r"[^a-z0-9\s]", "", text)        # remove punctuation and emojis

def score_sentiment(text):
    words = set(text.split())
    score = len(words & POSITIVE) - len(words & NEGATIVE)
    label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return label, score

def review_stream(rows):
    """Generator: yields cleaned Review objects and skips empty reviews."""
    for order_id, category, text in rows:
        cleaned = clean(text)
        if not cleaned:
            continue
        label, score = score_sentiment(cleaned)
        yield Review(order_id, category, cleaned, label, score)

reviews = list(review_stream(raw_reviews))
df = pd.DataFrame([asdict(r) for r in reviews])
print(df.to_string())

print("\nSentiment by category:")
print(pd.crosstab(df["category"], df["sentiment"]))

# Export negative reviews as JSON - ready to send to an LLM for root-cause summary
negatives = df[df["sentiment"] == "negative"].to_dict(orient="records")
with open("negative_reviews.json", "w") as f:
    json.dump(negatives, f, indent=2)
print(f"\nSaved {len(negatives)} negative reviews to negative_reviews.json")

> **Discussion**
>
> The keyword approach gives "Okay product, average battery life" a neutral score, which is right, but it would fail on "not great". This weakness is why we move to machine learning (Module 4) and deep learning (Module 5), where models learn sentiment from data instead of fixed word lists.

# Module 2: Statistics and Probability (25 min)

Statistics turns raw data into understanding and tells us whether a pattern is real or just noise. Every ML model is built on these ideas: loss functions are averages, model outputs are probabilities, and model comparisons are hypothesis tests.

## 2.1 Types of data

The data type decides which statistics and charts make sense.

| Type | Sub-type | Meaning | E-commerce example |
|---|---|---|---|
| Numerical | Continuous | Any value in a range | Order value Rs 1,449.50, time on site 7.3 min |
| Numerical | Discrete | Countable whole numbers | Quantity ordered, number of previous orders |
| Categorical | Nominal | Labels with no order | Payment method, city, product category |
| Categorical | Ordinal | Labels with an order | Rating 1-5, city tier 1/2/3, size S/M/L |


For ML, numerical columns are usually scaled, nominal columns are one-hot encoded, and ordinal columns can be mapped to ordered numbers.

## 2.2 Central tendency, spread and outliers

**Central tendency** describes the "typical" value:

- **Mean** = sum of values / number of values. Sensitive to outliers.
- **Median** = middle value of the sorted data. Robust to outliers.
- **Mode** = most frequent value. Useful for categorical data.

**Spread** describes how far values are from the centre:

- **Range** = max - min.
- **Variance** = average squared distance from the mean; **standard deviation** = square root of variance (same units as the data).
- **Percentiles / quartiles**: Q1 = 25th percentile, Q3 = 75th percentile, **IQR** = Q3 - Q1 covers the middle 50% of data.
- **Outlier rule (IQR)**: values below Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR.
- **Z-score** = (x - mean) / std: how many standard deviations a value is from the mean. |z| > 3 is unusual.

> **Real-world example**
>
> A store's daily orders are mostly Rs 450 to Rs 700, but one corporate bulk order is Rs 25,000. The mean jumps to about Rs 2,197 while the median stays at Rs 590. Reporting the mean would give the business a wrong picture of a "typical" order. This is the same as the primer's salary example, where the CEO's salary pulls the mean up.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from scipy import stats

# Real-world example: order values (Rs) from one day at a small online store.
# The last order is a bulk corporate purchase - an outlier.
order_values = np.array([450, 520, 610, 480, 700, 650, 530, 590, 610, 560,
                         620, 610, 480, 540, 25000])

print("Mean   :", round(order_values.mean(), 2))
print("Median :", np.median(order_values))
print("Mode   :", stats.mode(order_values, keepdims=False).mode)

# Spread
print("\nRange             :", order_values.max() - order_values.min())
print("Variance (sample) :", round(order_values.var(ddof=1), 2))
print("Std dev  (sample) :", round(order_values.std(ddof=1), 2))

q1, q3 = np.percentile(order_values, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, outlier fences=({lower}, {upper})")
print("Outliers (IQR rule):", order_values[(order_values < lower) | (order_values > upper)])

# Z-score: how many standard deviations a value is from the mean
z = (order_values - order_values.mean()) / order_values.std(ddof=1)
print("Z-score of the Rs 25,000 order:", round(z[-1], 2))

# Effect of removing the outlier
clean = order_values[order_values <= upper]
print(f"\nWithout outlier -> mean {clean.mean():.2f}, median {np.median(clean)}")

# Pandas gives everything in one line
print("\n", pd.Series(order_values, name="order_value").describe().round(2))

## 2.3 Shape of a distribution and visualisation

- **Skewness** measures asymmetry. Right-skewed (positive) data has a long tail of large values: income, order value, house prices. For right-skewed data, mean > median.
- **Kurtosis** measures how heavy the tails are. High kurtosis means extreme values are more common than in a normal distribution.
- **Histogram**: shows the shape of one numeric variable.
- **Box plot**: shows median, quartiles and outliers at a glance.
- **Scatter plot**: shows the relationship between two numeric variables; the **correlation coefficient** (-1 to +1) measures the strength of a linear relationship.

![Figure 3: The 68-95-99.7 rule of the normal distribution and the parts of a box plot.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d03_normal_boxplot.png)
*Figure 3: The 68-95-99.7 rule of the normal distribution and the parts of a box plot.*


> **Real-world example**
>
> Order values on Indian e-commerce sites are strongly right-skewed. Analysts therefore report median order value, and ML engineers often apply a log transform (`np.log1p`) before modelling. We do exactly this in the final project.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(1)
# Order values are typically right-skewed: many small orders, a few large ones
order_values = rng.lognormal(mean=7, sigma=0.6, size=2000)
# Time on site vs order value (positive relationship + noise)
time_on_site = rng.normal(15, 5, 2000).clip(1)
spend = 200 + 60 * time_on_site + rng.normal(0, 250, 2000)

print("Skewness :", round(stats.skew(order_values), 2), "(> 0 means right-skewed)")
print("Kurtosis :", round(stats.kurtosis(order_values), 2), "(> 0 means heavier tails than normal)")
print("Mean vs median:", round(order_values.mean()), "vs", round(np.median(order_values)))
print("Correlation (time on site vs spend):", round(np.corrcoef(time_on_site, spend)[0, 1], 2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(order_values, bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(order_values.mean(), color="red", label="Mean")
axes[0].axvline(np.median(order_values), color="green", label="Median")
axes[0].set_title("Histogram: order value (right-skewed)")
axes[0].legend()

axes[1].boxplot(order_values)
axes[1].set_title("Box plot: dots beyond whiskers are outliers")

axes[2].scatter(time_on_site, spend, s=5, alpha=0.4)
axes[2].set_xlabel("Minutes on site")
axes[2].set_ylabel("Spend (Rs)")
axes[2].set_title("Scatter: time on site vs spend")
plt.tight_layout()
plt.show()

## 2.4 Probability, conditional probability and Bayes' theorem

Probability is a number between 0 and 1 describing how likely an event is. **Conditional probability** P(A | B) is the probability of A given that B has happened. **Bayes' theorem** updates a prior belief with new evidence:

**P(A | B) = P(B | A) x P(A) / P(B)**

- P(A) is the prior (before evidence), P(B | A) is the likelihood, P(A | B) is the posterior (after evidence).
- When the event is rare, even an accurate test produces many false alarms. This is called the base-rate effect.

> **Real-world example**
>
> A fraud model catches 95% of fraud and wrongly flags only 5% of genuine payments. It sounds excellent, but only 1% of payments are fraud. Bayes' theorem shows that only about 16% of flagged payments are actually fraud. The business must plan manual review capacity accordingly. LLMs also work with conditional probability: they predict P(next word | previous words).

In [ ]:
import numpy as np

# Real-world example: a fraud model flags a payment. How likely is it really fraud?
p_fraud = 0.01                 # prior: 1% of payments are fraudulent
p_flag_given_fraud = 0.95      # sensitivity: model catches 95% of fraud
p_flag_given_genuine = 0.05    # false-alarm rate on genuine payments

# Total probability of a flag
p_flag = p_flag_given_fraud * p_fraud + p_flag_given_genuine * (1 - p_fraud)

# Bayes' theorem: P(fraud | flag) = P(flag | fraud) * P(fraud) / P(flag)
p_fraud_given_flag = p_flag_given_fraud * p_fraud / p_flag
print(f"P(flag)          = {p_flag:.4f}")
print(f"P(fraud | flag)  = {p_fraud_given_flag:.3f}")
print(f"-> only ~{p_fraud_given_flag:.0%} of flagged payments are real fraud")

# Verify with a simulation of 1,000,000 payments
rng = np.random.default_rng(0)
n = 1_000_000
is_fraud = rng.random(n) < p_fraud
flagged = np.where(is_fraud, rng.random(n) < p_flag_given_fraud, rng.random(n) < p_flag_given_genuine)
print(f"Simulated P(fraud | flag) = {is_fraud[flagged].mean():.3f}")

# Conditional probability from a table: P(return | COD) vs P(return | prepaid)
cod_orders, cod_returns = 2000, 520
prepaid_orders, prepaid_returns = 8000, 1200
print(f"\nP(return | COD)     = {cod_returns / cod_orders:.2f}")
print(f"P(return | prepaid) = {prepaid_returns / prepaid_orders:.2f}")

## 2.5 Probability distributions

A probability distribution describes how likely each value of a random variable is. The **expected value** E[X] is the long-run average.

| Distribution | Type | Models | Example |
|---|---|---|---|
| Bernoulli / Binomial | Discrete | Number of successes in n yes/no trials | How many of 20 visitors buy |
| Geometric | Discrete | Trials until the first success | Visitors until the first sale |
| Poisson | Discrete | Count of events in a fixed interval | Orders arriving per minute |
| Uniform | Continuous | All values equally likely | Random A/B group assignment |
| Normal (Gaussian) | Continuous | Symmetric bell curve | Delivery time, measurement errors |
| Exponential | Continuous | Time between events | Seconds between two orders |


The **Central Limit Theorem (CLT)** says that the average of many independent samples is approximately normally distributed, even when the original data is skewed. This is why confidence intervals and t-tests work on real-world data.

> **Real-world example**
>
> Capacity planning for a flash sale: if orders arrive at 4 per minute on average (Poisson), the probability of more than 8 orders in a minute is about 2%. The engineering team uses such numbers to size servers and payment gateways.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Binomial: 20 visitors, each buys with probability 0.10
binom = stats.binom(n=20, p=0.10)
print(f"Binomial  P(exactly 2 buy)   = {binom.pmf(2):.3f}")
print(f"Binomial  P(at least 3 buy)  = {1 - binom.cdf(2):.3f}")

# Geometric: how many visitors until the first purchase?
geom = stats.geom(p=0.10)
print(f"Geometric P(first sale within 5 visitors) = {geom.cdf(5):.3f}")

# Poisson: on average 4 orders arrive per minute
pois = stats.poisson(mu=4)
print(f"Poisson   P(more than 8 orders in a minute) = {1 - pois.cdf(8):.4f}")

# Normal: delivery time ~ N(mean=3 days, sd=1 day)
norm = stats.norm(loc=3, scale=1)
print(f"Normal    P(delivery > 5 days) = {1 - norm.cdf(5):.4f}")
print(f"Normal    95% of deliveries fall between {norm.ppf(0.025):.2f} and {norm.ppf(0.975):.2f} days")

# Exponential: time between orders, average 15 seconds
expo = stats.expon(scale=15)
print(f"Exponential P(next order within 5 s) = {expo.cdf(5):.3f}")

# Central Limit Theorem: averages of a skewed variable become normal
rng = np.random.default_rng(0)
population = rng.exponential(scale=15, size=100_000)          # very skewed
sample_means = [rng.choice(population, 50).mean() for _ in range(3000)]

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
k = np.arange(0, 11)
axes[0, 0].bar(k, binom.pmf(k)); axes[0, 0].set_title("Binomial(n=20, p=0.1)")
axes[0, 1].bar(k + 1, geom.pmf(k + 1)); axes[0, 1].set_title("Geometric(p=0.1)")
axes[0, 2].bar(np.arange(0, 13), pois.pmf(np.arange(0, 13))); axes[0, 2].set_title("Poisson(lambda=4)")
x = np.linspace(-1, 7, 200)
axes[1, 0].plot(x, norm.pdf(x)); axes[1, 0].set_title("Normal(3, 1)")
axes[1, 1].hist(population, bins=60, color="orange"); axes[1, 1].set_title("Skewed population")
axes[1, 2].hist(sample_means, bins=40, color="green")
axes[1, 2].set_title("CLT: means of samples (n=50)")
plt.tight_layout()
plt.show()

## 2.6 Statistical inference: confidence intervals and hypothesis tests

We rarely see the whole population, so we estimate from samples.

- **Confidence interval (CI)**: a range that contains the true value with a stated confidence (usually 95%). CI = estimate +/- critical value x standard error.
- **Null hypothesis H0**: "no effect / no difference". **Alternative H1**: "there is an effect".
- **p-value**: the probability of seeing data this extreme if H0 were true. If p < alpha (0.05), reject H0.
- **Type I error**: false alarm (rejecting a true H0). **Type II error**: missing a real effect.

![Figure 4: The five steps of a hypothesis test.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d04_hypothesis.png)
*Figure 4: The five steps of a hypothesis test.*


| Test | Use it when | Example question |
|---|---|---|
| Z-test / two-proportion z-test | Comparing proportions or means with large samples | Is conversion of page B higher than page A? |
| One-sample t-test | Comparing a sample mean with a fixed value | Is average delivery time different from the promised 3 days? |
| Two-sample (Welch) t-test | Comparing means of two groups | Do members spend more than non-members? |
| Chi-square test | Relationship between two categorical variables | Is payment method related to returns? |
| ANOVA | Comparing means of three or more groups | Does average rating differ across App, Web, Desktop? |

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from scipy import stats

rng = np.random.default_rng(10)

# 1) 95% confidence interval for the average order value
sample = rng.normal(1450, 380, 200)           # 200 sampled orders
mean, se = sample.mean(), stats.sem(sample)
ci_low, ci_high = stats.t.interval(0.95, df=len(sample) - 1, loc=mean, scale=se)
print(f"Average order value = {mean:.0f}, 95% CI = ({ci_low:.0f}, {ci_high:.0f})")

# 2) One-sample t-test: is the average delivery time different from the promised 3 days?
delivery = rng.normal(3.3, 1.1, 120)
t, p = stats.ttest_1samp(delivery, popmean=3)
print(f"\nOne-sample t-test: t={t:.2f}, p={p:.4f} ->",
      "Reject H0: delivery differs from 3 days" if p < 0.05 else "Fail to reject H0")

# 3) Two-sample (Welch) t-test: do members spend more than non-members?
members = rng.normal(1600, 400, 150)
non_members = rng.normal(1450, 420, 180)
t, p = stats.ttest_ind(members, non_members, equal_var=False)
print(f"Two-sample t-test: t={t:.2f}, p={p:.4f} ->",
      "Members spend differently" if p < 0.05 else "No evidence of a difference")

# 4) Chi-square test of independence: is payment method related to returns?
table = pd.DataFrame({"Returned": [260, 310, 120], "Kept": [740, 1690, 880]},
                     index=["COD", "UPI", "Card"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"\nChi-square: chi2={chi2:.1f}, dof={dof}, p={p:.2e}")
print("Return rate by payment:\n", (table["Returned"] / table.sum(axis=1)).round(3))

# 5) One-way ANOVA: does the average rating differ across devices?
app = rng.normal(4.2, 0.6, 100)
web = rng.normal(4.0, 0.6, 100)
desktop = rng.normal(4.05, 0.6, 100)
f, p = stats.f_oneway(app, web, desktop)
print(f"\nANOVA: F={f:.2f}, p={p:.4f} ->",
      "At least one device differs" if p < 0.05 else "No significant difference")

## 2.7 Module 2 practical: A/B test of a new checkout page

The product team tested a new one-click checkout page. 12,000 visitors saw the old page (A) and 12,000 saw the new page (B). We test whether the higher conversion of B is real, estimate the uplift with a confidence interval, check basket value, and make a business decision.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# PRACTICAL: A/B test of a new one-click checkout page
# Version A = old page, Version B = new page. Visitors were split randomly.
visitors_a, buyers_a = 12_000, 384      # 3.20% conversion
visitors_b, buyers_b = 12_000, 456      # 3.80% conversion

p_a, p_b = buyers_a / visitors_a, buyers_b / visitors_b
print(f"Conversion A = {p_a:.2%}, Conversion B = {p_b:.2%}, uplift = {(p_b - p_a) / p_a:.1%}")

# Step 1: hypotheses.  H0: p_A = p_B   H1: p_A != p_B   (alpha = 0.05)
# Step 2: two-proportion z-test
p_pool = (buyers_a + buyers_b) / (visitors_a + visitors_b)
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / visitors_a + 1 / visitors_b))
z = (p_b - p_a) / se_pool
p_value = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"z = {z:.2f}, p-value = {p_value:.4f}")

# Step 3: 95% confidence interval for the difference
se_diff = np.sqrt(p_a * (1 - p_a) / visitors_a + p_b * (1 - p_b) / visitors_b)
low, high = (p_b - p_a) - 1.96 * se_diff, (p_b - p_a) + 1.96 * se_diff
print(f"95% CI for (B - A): {low:.2%} to {high:.2%}")

# Step 4: the same answer with a chi-square test
table = [[buyers_a, visitors_a - buyers_a], [buyers_b, visitors_b - buyers_b]]
chi2, p_chi, _, _ = stats.chi2_contingency(table, correction=False)
print(f"Chi-square p-value = {p_chi:.4f} (matches the z-test)")

# Step 5: did the average basket value change? (Welch t-test on simulated baskets)
rng = np.random.default_rng(3)
basket_a = rng.normal(1480, 400, buyers_a)
basket_b = rng.normal(1495, 410, buyers_b)
t, p_t = stats.ttest_ind(basket_b, basket_a, equal_var=False)
print(f"Basket value t-test p-value = {p_t:.3f}")

# Step 6: business decision
if p_value < 0.05 and low > 0:
    extra = (p_b - p_a) * 100_000 * basket_b.mean()
    print("\nDECISION: roll out version B.")
    print(f"Expected extra revenue per 1 lakh visitors ~ Rs {extra:,.0f}")
else:
    print("\nDECISION: keep version A - the difference is not significant.")

plt.bar(["A (old)", "B (new)"], [p_a * 100, p_b * 100], color=["grey", "seagreen"],
        yerr=[1.96 * np.sqrt(p_a * (1 - p_a) / visitors_a) * 100,
              1.96 * np.sqrt(p_b * (1 - p_b) / visitors_b) * 100], capsize=8)
plt.ylabel("Conversion rate (%)")
plt.title("A/B test result with 95% error bars")
plt.show()

> **Interpretation**
>
> p = 0.011 < 0.05 and the 95% CI for the difference (0.14% to 1.06%) is entirely above zero, so the new page really converts better. Basket value did not change (p = 0.49), so the gain comes from more buyers, not bigger baskets. The same method is used to compare two prompts or two LLM versions in production.

# Module 3: Linear Algebra for ML and GenAI (20 min)

Linear algebra is the language of data and neural networks. A dataset is a matrix, a word or product is represented as a vector (embedding), a neural network layer is a matrix multiplication, and semantic search is a dot product.

## 3.1 Scalars, vectors, matrices and tensors

| Object | Dimensions | NumPy shape | Deep learning use |
|---|---|---|---|
| Scalar | 0-D | () | A single loss value |
| Vector | 1-D | (n,) | One customer's features, a bias vector, a word embedding |
| Matrix | 2-D | (rows, cols) | A dataset (samples x features), a layer's weight matrix |
| Tensor | 3-D and more | (batch, channels, height, width) | A batch of images or a batch of token sequences |


The **shape** of an array is the most important thing to check when debugging ML code. Most deep learning errors are shape mismatches.

In [ ]:
import numpy as np

scalar = np.array(5)                                   # 0-D: e.g. one loss value
vector = np.array([1, 2, 3])                           # 1-D: e.g. one customer's features
matrix = np.array([[1, 2, 3], [4, 5, 6]])              # 2-D: e.g. a table of customers x features
tensor = np.zeros((32, 3, 224, 224))                   # 4-D: a batch of 32 RGB images, 224x224

for name, obj in [("Scalar", scalar), ("Vector", vector), ("Matrix", matrix), ("Tensor", tensor)]:
    print(f"{name:7}: ndim={obj.ndim}, shape={obj.shape}")

# Real-world example: 4 customers described by [age, orders, avg_spend]
customers = np.array([[25, 3, 1200],
                      [41, 12, 3400],
                      [33, 7, 2100],
                      [19, 1, 600]])
print("\nCustomer matrix shape:", customers.shape, "-> 4 rows (samples) x 3 columns (features)")
print("Second customer vector:", customers[1])
print("All ages (column 0):   ", customers[:, 0])

# Reshape: the same data viewed with a different shape (common before feeding a model)
flat = customers.reshape(-1)          # -1 means "work it out for me"
print("Flattened:", flat, "shape", flat.shape)
print("Back to 2 x 6:\n", flat.reshape(2, 6))

## 3.2 Vector operations, norms, dot product and cosine similarity

- **Addition / subtraction**: element-wise. **Scalar multiplication** stretches a vector without changing its direction.
- **Norm** = length of a vector. L1 (Manhattan) = sum of absolute values; L2 (Euclidean) = square root of sum of squares; Infinity norm = largest absolute value. L1 and L2 norms are also used for regularisation (Lasso and Ridge / weight decay).
- **Dot product** u . v = sum of u_i x v_i. It measures how much two vectors point in the same direction. A neuron computes a dot product between inputs and weights.
- **Cosine similarity** = (u . v) / (||u|| x ||v||). It ignores length and measures only direction: 1 = same direction, 0 = unrelated (orthogonal), -1 = opposite.

Worked example from the primer: u = [3, 4], v = [6, 0]. u . v = 18, ||u|| = 5, ||v|| = 6, so cosine similarity = 18 / 30 = **0.6**.

![Figure 5: Cosine similarity compares directions of embedding vectors (left); each entry of a matrix product is a row-column dot product (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d05_vectors_matrix.png)
*Figure 5: Cosine similarity compares directions of embedding vectors (left); each entry of a matrix product is a row-column dot product (right).*


> **Real-world example**
>
> Embedding models turn every product, document or sentence into a vector of hundreds of numbers. Similar items get vectors that point in similar directions. "Customers who viewed earbuds also viewed..." and every vector database (Pinecone, FAISS, Snowflake Cortex Search) rank results by cosine similarity.

In [ ]:
import numpy as np

u = np.array([3, 4])
v = np.array([6, 0])

print("u + v =", u + v)                 # [9 4]
print("u - v =", u - v)                 # [-3 4]
print("2u    =", 2 * u)                 # [6 8]

# Norms (vector length)
x = np.array([3, -4, 12])
print("\nL1 norm  :", np.linalg.norm(x, 1))        # |3|+|-4|+|12| = 19
print("L2 norm  :", np.linalg.norm(x))           # sqrt(9+16+144) = 13
print("Inf norm :", np.linalg.norm(x, np.inf))   # max |x_i| = 12

# Dot product (example from the primer)
x = np.array([2, -1, 5])
w = np.array([1, 3, -2])
print("\nx . w =", np.dot(x, w), "| same as x @ w =", x @ w)    # -11

# Cosine similarity (example from the primer): u=[3,4], v=[6,0] -> 0.6
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("cos(u, v) =", round(cosine_similarity(u, v), 2))

# Real-world example: toy 4-D "embeddings" of products
# dims ~ [electronics-ness, fashion-ness, sports-ness, price level]
emb = {
    "wireless earbuds": np.array([0.90, 0.05, 0.20, 0.40]),
    "bluetooth speaker": np.array([0.85, 0.02, 0.10, 0.45]),
    "running shoes":     np.array([0.05, 0.60, 0.80, 0.50]),
    "yoga mat":          np.array([0.02, 0.10, 0.90, 0.20]),
    "denim jeans":       np.array([0.01, 0.95, 0.05, 0.40]),
}
query = "wireless earbuds"
scores = {name: cosine_similarity(emb[query], vec) for name, vec in emb.items() if name != query}
print(f"\nProducts most similar to '{query}':")
for name, s in sorted(scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {name:18} {s:.3f}")

## 3.3 Matrix operations and solving linear systems

- **Matrix multiplication** C = AB: entry C[i, j] is the dot product of row i of A and column j of B. The inner dimensions must match: (m x n)(n x p) gives (m x p). Order matters: AB is usually not equal to BA.
- **Transpose** A^T flips rows and columns. It is used in backpropagation.
- **Inverse** A^-1 "undoes" A, so A A^-1 = I (identity). Only square matrices with a non-zero **determinant** have an inverse (a zero determinant means a singular matrix).
- **System of equations** Ax = b: `np.linalg.solve` finds x. Deep learning avoids direct inversion for huge matrices and uses gradient descent instead.
- **Broadcasting**: NumPy automatically stretches a smaller array (e.g. a bias vector) to match a larger one.

> **Real-world example**
>
> Three customers bought different combinations of T-shirts, jeans and caps, and we only know their bill totals. Writing quantities as a matrix and bills as a vector, solving Ax = b recovers the unit price of each product. Matrix multiplication then prices many new baskets in one step.

In [ ]:
import numpy as np

A = np.array([[1, 2], [3, 4]])
B = np.array([[2, 0], [1, 2]])
print("A @ B =\n", A @ B)                 # [[4 4] [10 8]]  (example from the primer)
print("B @ A =\n", B @ A, "\n(order matters: AB != BA)")
print("A transpose =\n", A.T)
print("det(A) =", round(np.linalg.det(A), 2))
A_inv = np.linalg.inv(A)
print("A inverse =\n", A_inv)
print("A @ A_inv =\n", np.round(A @ A_inv, 10), "(identity matrix)")

# Real-world example: find unit prices from three customers' bills (Ax = b)
# Customer 1 bought 2 tees, 1 jeans, 1 cap  -> Rs 3,096
# Customer 2 bought 1 tee, 2 jeans, 0 caps  -> Rs 3,497
# Customer 3 bought 1 tee, 0 jeans, 3 caps  -> Rs 2,296
quantities = np.array([[2, 1, 1],
                       [1, 2, 0],
                       [1, 0, 3]])
bills = np.array([3096, 3497, 2296])
unit_prices = np.linalg.solve(quantities, bills)
print("\nUnit prices [tee, jeans, cap]:", unit_prices.round(2))

# Matrix multiplication for many baskets at once
baskets = np.array([[1, 1, 0], [0, 1, 1], [4, 0, 0]])   # 3 new baskets
print("Basket totals:", baskets @ unit_prices)

# Broadcasting: apply a different discount to each product column in one step
discount = np.array([0.10, 0.20, 0.00])
print("Discounted prices per basket:\n", baskets * unit_prices * (1 - discount))

## 3.4 Eigenvalues, SVD and PCA

- An **eigenvector** v of a square matrix A keeps its direction when multiplied by A; it is only stretched by the **eigenvalue** lambda: **Av = lambda v**.
- **Singular Value Decomposition (SVD)** factors any matrix into rotation x scaling x rotation: **A = U Sigma V^T**. Keeping only the largest singular values gives the best low-rank approximation of the matrix. This is used for recommendations, compression and LoRA fine-tuning of LLMs (low-rank weight updates).
- **Principal Component Analysis (PCA)** uses eigenvectors of the covariance matrix to find the directions of maximum variance and compresses many features into a few.

> **Real-world example**
>
> A users x products rating matrix has many empty cells. SVD finds hidden "taste" patterns (for example, an electronics lover vs a fitness lover) and fills in predicted ratings, which drive "Recommended for you". PCA is used to visualise high-dimensional embeddings in 2-D.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# 1) Eigenvalues and eigenvectors: A v = lambda v
A = np.array([[2, 1], [1, 2]])
eigenvalues, eigenvectors = np.linalg.eigh(A)    # eigh = for symmetric matrices
print("Eigenvalues:", eigenvalues)
v = eigenvectors[:, 1]                            # eigenvector for the largest eigenvalue
print("A @ v      :", A @ v)
print("lambda * v :", eigenvalues[1] * v, "(same direction, only scaled)")

# 2) SVD for recommendations: users x products rating matrix (0 = not rated)
ratings = np.array([[5, 4, 0, 1, 0],
                    [4, 5, 1, 0, 1],
                    [1, 0, 5, 4, 5],
                    [0, 1, 4, 5, 4],
                    [5, 5, 0, 0, 1]], dtype=float)
U, S, Vt = np.linalg.svd(ratings)
print("\nSingular values:", S.round(2))
k = 2                                             # keep the 2 strongest "taste" patterns
approx = U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]
print("Rank-2 reconstruction (predicted ratings):\n", approx.round(1))
print("Predicted score of user 0 for product 2:", approx[0, 2].round(2), "(low -> do not recommend)")

# 3) PCA on a real dataset: 13 chemical features of wines -> 2 dimensions
wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
pca = PCA(n_components=2)
X2 = pca.fit_transform(X)
print("\nExplained variance by PC1, PC2:", pca.explained_variance_ratio_.round(3),
      "| total:", pca.explained_variance_ratio_.sum().round(3))

for label in np.unique(wine.target):
    plt.scatter(X2[wine.target == label, 0], X2[wine.target == label, 1],
                label=wine.target_names[label], alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.legend()
plt.title("PCA: 13 features compressed to 2")
plt.show()

## 3.5 Linear algebra inside a neural network

A dense (fully connected) layer does three things: rotate/scale the input with a weight matrix W, shift it with a bias vector b, and bend it with a non-linear activation:

**Forward pass: Z = XW + b, then A = activation(Z)**

With a batch of 2 samples and 3 features (X is 2 x 3) and a layer of 4 neurons (W is 3 x 4), the output is 2 x 4. During training, the transpose W^T carries error gradients backward.

In [ ]:
import numpy as np

# One dense neural-network layer is just linear algebra: Z = XW + b, A = ReLU(Z)
rng = np.random.default_rng(0)
X = np.array([[0.5, 1.2, -0.3],          # 2 samples (batch), 3 input features each
              [1.0, -0.7, 2.0]])
W = rng.normal(0, 0.5, size=(3, 4))      # 3 inputs -> 4 neurons
b = np.zeros(4)                          # one bias per neuron

Z = X @ W + b                            # (2x3) @ (3x4) + (4,) -> (2x4), bias is broadcast
A = np.maximum(0, Z)                     # ReLU activation
print("X shape", X.shape, "| W shape", W.shape, "| output shape", A.shape)
print("Z (before activation):\n", Z.round(3))
print("A (after ReLU):\n", A.round(3))

## 3.6 Module 3 practical: mini semantic product search

This is the retrieval step of a RAG system built only with linear algebra. Each product description becomes a TF-IDF vector (a row of a matrix). The user's query becomes a vector in the same space, and products are ranked by cosine similarity. In production, TF-IDF is replaced by neural embeddings, but the maths is identical.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# PRACTICAL: mini product search engine = the "retrieval" step of RAG
catalog = pd.DataFrame({
    "product": ["Wireless Earbuds", "Bluetooth Speaker", "Running Shoes", "Yoga Mat",
                "Denim Jeans", "Air Fryer", "Mixer Grinder", "Face Serum", "Sunscreen SPF50",
                "Smart Watch"],
    "description": [
        "bluetooth wireless earbuds with noise cancellation and long battery life",
        "portable bluetooth speaker with deep bass and waterproof body",
        "lightweight running shoes with cushioned sole for jogging and gym",
        "non slip yoga mat for home workout, gym and stretching",
        "slim fit blue denim jeans for men, stretchable cotton fabric",
        "air fryer for oil free cooking, crispy fries and snacks",
        "powerful mixer grinder with three jars for chutney and juice",
        "vitamin c face serum for glowing skin and dark spots",
        "sunscreen spf 50 for skin protection from sun, non greasy",
        "smart watch with heart rate, step tracking and bluetooth calling",
    ],
})

# Step 1: turn every description into a vector (a matrix: products x vocabulary)
vectorizer = TfidfVectorizer(stop_words="english")
doc_vectors = vectorizer.fit_transform(catalog["description"])
print("Document-term matrix shape:", doc_vectors.shape)

# Step 2: turn the user's query into a vector and rank products by cosine similarity
def search(query, top_k=3):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_vectors).ravel()
    best = [i for i in np.argsort(scores)[::-1][:top_k] if scores[i] > 0]
    return catalog.iloc[best].assign(score=scores[best].round(3))[["product", "score"]]

for q in ["something for my gym workout", "bluetooth music", "skin care for sun"]:
    print(f"\nQuery: '{q}'")
    print(search(q).to_string(index=False))

# Step 3: verify the library with our own NumPy cosine formula
q = vectorizer.transform(["bluetooth music"]).toarray()[0]
d = doc_vectors[1].toarray()[0]
manual = q @ d / (np.linalg.norm(q) * np.linalg.norm(d))
print(f"\nManual cosine for 'Bluetooth Speaker' = {manual:.3f}")

> **Discussion**
>
> TF-IDF only matches words that literally appear. The query "something for my gym workout" finds the yoga mat and running shoes because "gym" and "workout" appear in their descriptions, but a query like "fitness gear" would find nothing. Neural embeddings (Module 5) understand meaning, so "fitness" and "gym" end up close together.

# Module 4: Machine Learning (40 min)

Machine Learning (ML) is the science of letting computers learn patterns from data instead of following hand-written rules. Given examples of inputs and outputs, an ML algorithm finds a function that maps inputs to outputs and generalises to new, unseen data.

## 4.1 Types of machine learning

![Figure 6: The three families of machine learning with business examples.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d06_ml_types.png)
*Figure 6: The three families of machine learning with business examples.*


- **Supervised learning**: data has labels (the right answer). **Regression** predicts a number (delivery time, sales); **classification** predicts a category (returned or not, spam or not).
- **Unsupervised learning**: no labels. The algorithm finds structure on its own: **clustering** (customer segments) and **dimensionality reduction** (PCA).
- **Reinforcement learning**: an agent takes actions and learns from rewards. RLHF (Reinforcement Learning from Human Feedback) is how chat LLMs are aligned with human preferences.

Key vocabulary: **features** (X) are the input columns, the **target** or **label** (y) is what we predict, a **model** is the learned function, **training** is fitting the model to data, and **inference** is using the model on new data.

## 4.2 The ML workflow

![Figure 7: The end-to-end machine learning workflow. Data preparation usually takes most of the effort.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d07_ml_workflow.png)
*Figure 7: The end-to-end machine learning workflow. Data preparation usually takes most of the effort.*


1. Define the business problem and success metric. 2. Collect data. 3. Clean data and explore it (EDA). 4. Engineer features. 5. Split into train and test sets. 6. Train several models. 7. Evaluate and tune. 8. Deploy and monitor for drift.

## 4.3 Data preprocessing

Models need clean, numeric input on comparable scales.

- **Missing values**: impute with median (numeric, robust to outliers) or most frequent value (categorical).
- **Encoding**: **one-hot encoding** turns a categorical column into 0/1 columns, one per category.
- **Scaling**: **StandardScaler** converts a column to mean 0, std 1. Distance-based and gradient-based models (KNN, SVM, logistic regression, neural networks) need it; tree models do not.
- **Train/test split**: hold back 20 to 30% of data to measure performance on unseen data. Use `stratify=y` for classification so both sets have the same class balance.
- **Data leakage**: never fit imputers or scalers on the test set. A `Pipeline` with a `ColumnTransformer` guarantees that preprocessing is learned from training data only.

> **Real-world example**
>
> A churn dataset has missing ages, a missing device value and text columns (city tier, device). The pipeline below fills, encodes and scales them in a repeatable way, so the exact same steps run in training and in production.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Real-world example: raw customer data with missing values and text categories
df = pd.DataFrame({
    "age":       [25, 41, np.nan, 19, 35, 52, 28, np.nan, 44, 31],
    "city_tier": ["Tier1", "Tier2", "Tier1", "Tier3", "Tier1",
                  "Tier2", "Tier3", "Tier1", "Tier2", "Tier1"],
    "device":    ["App", "Web", "App", np.nan, "Desktop", "App", "Web", "App", "Desktop", "App"],
    "orders":    [3, 12, 7, 1, 9, 15, 2, 5, 11, 6],
    "avg_spend": [1200, 3400, 2100, 600, 2800, 4100, 750, 1500, 3000, 1900],
    "churned":   [1, 0, 0, 1, 0, 0, 1, 1, 0, 0],
})
print("Missing values per column:\n", df.isna().sum())

X = df.drop(columns="churned")
y = df["churned"]

# 1) Split FIRST so the test set never influences preprocessing (avoids data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 2) Different treatment for numeric and categorical columns
numeric_cols = ["age", "orders", "avg_spend"]
categorical_cols = ["city_tier", "device"]

numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                         ("scale", StandardScaler())])
categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                             ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_cols),
                                  ("cat", categorical_pipe, categorical_cols)])

# 3) fit on train only, then transform both
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print("\nFeatures after preprocessing:", list(feature_names))
print(pd.DataFrame(X_train_ready, columns=feature_names).round(2).to_string())
print("\nTrain shape:", X_train_ready.shape, "| Test shape:", X_test_ready.shape)

## 4.4 Regression

Regression predicts a continuous number.

- **Linear regression** fits y = w1x1 + w2x2 + ... + b by minimising the squared error. It is simple and interpretable: each coefficient is the change in y for a one-unit change in that feature, holding the others constant.
- **Random Forest regression** averages many decision trees and captures non-linear patterns and interactions.

| Metric | Formula idea | Interpretation |
|---|---|---|
| MAE | Average of absolute errors | Average miss, in the target's units |
| RMSE | Square root of average squared error | Like MAE but punishes big misses more |
| R-squared | 1 - (model error / variance of y) | Share of variation explained (1 = perfect, 0 = no better than the mean) |


> **Real-world example**
>
> Property portals estimate house prices from income, location and house features. We use the real California Housing dataset (20,640 districts). Random Forest explains about 80% of price variation versus about 58% for linear regression, because location effects are non-linear.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Real dataset: 20,640 California districts. Target = median house value (in $100,000s)
data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.head())

X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name:18} MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}")
    return pred

lin_pred = evaluate("Linear Regression", LinearRegression())
rf_pred = evaluate("Random Forest", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))

# Interpreting linear regression: each coefficient = change in price per unit of the feature
coefs = pd.Series(LinearRegression().fit(X_train, y_train).coef_, index=X.columns).sort_values()
print("\nLinear regression coefficients:\n", coefs.round(3))

plt.figure(figsize=(6, 6))
plt.scatter(y_test, rf_pred, s=4, alpha=0.3)
plt.plot([0, 5], [0, 5], "r--", label="Perfect prediction")
plt.xlabel("Actual value"); plt.ylabel("Predicted value")
plt.title("Random Forest: actual vs predicted"); plt.legend()
plt.show()

## 4.5 Classification and evaluation metrics

Classification predicts a category. Popular algorithms:

- **Logistic regression**: a linear model passed through the sigmoid function to output a probability. Fast, interpretable, strong baseline.
- **Decision tree**: a sequence of if-else splits learned from data. Easy to explain but overfits easily.
- **Random Forest**: many trees on random samples of rows and features, with a majority vote. Robust and accurate.
- **Gradient boosting**: trees built one after another, each fixing the previous errors (XGBoost, LightGBM). Often the best on tabular data.

![Figure 8: Underfitting, good fit and overfitting (left three panels); the confusion matrix and its metrics (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d08_fit_confusion.png)
*Figure 8: Underfitting, good fit and overfitting (left three panels); the confusion matrix and its metrics (right).*


| Metric | Formula | Ask yourself |
|---|---|---|
| Accuracy | (TP + TN) / all | What share of predictions is correct? (misleading when classes are imbalanced) |
| Precision | TP / (TP + FP) | When the model says "yes", how often is it right? |
| Recall | TP / (TP + FN) | Of all real "yes" cases, how many did we catch? |
| F1-score | Harmonic mean of precision and recall | Balance of both |
| ROC-AUC | Area under the ROC curve | How well does the model rank positives above negatives (0.5 = random, 1 = perfect)? |


> **Real-world example**
>
> In cancer screening, a false negative (missing a malignant tumour) is far worse than a false alarm, so recall for the malignant class matters most. In spam filtering, precision matters more, because sending a genuine email to spam is costly. We use the real Wisconsin Breast Cancer dataset (569 samples, 30 measurements).

In [ ]:
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

# Real dataset: 569 tumour samples, 30 measurements each. Target: 0 = malignant, 1 = benign
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, pred),
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred),
                 "f1": f1_score(y_test, pred),
                 "roc_auc": roc_auc_score(y_test, proba)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

best = models["Logistic Regression"]
pred = best.predict(X_test)
print("\nConfusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=data.target_names))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=data.target_names, cmap="Blues")
plt.title("Logistic Regression - confusion matrix")
plt.show()

## 4.6 Overfitting, cross-validation and hyper-parameter tuning

- **Underfitting** (high bias): the model is too simple and performs poorly on both train and test data.
- **Overfitting** (high variance): the model memorises noise. Training score is very high but test score is lower.
- **Fixes for overfitting**: more data, simpler models (limit tree depth), regularisation (L1/L2), dropout and early stopping (deep learning).
- **K-fold cross-validation**: split data into K parts, train K times, each time testing on a different part, and average the scores. More reliable than one split.
- **GridSearchCV** tries every combination of hyper-parameters (settings chosen before training, such as tree depth) with cross-validation and keeps the best.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

# 1) Overfitting: as the tree gets deeper, train accuracy -> 100% but test accuracy stalls
depths = range(1, 16)
train_acc, test_acc = [], []
for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append(tree.score(X_test, y_test))

plt.plot(depths, train_acc, "o-", label="Train accuracy")
plt.plot(depths, test_acc, "s-", label="Test accuracy")
plt.xlabel("Tree depth (model complexity)"); plt.ylabel("Accuracy")
plt.title("Underfitting (left) vs overfitting (right)"); plt.legend()
plt.show()

# 2) 5-fold cross-validation: a more reliable estimate than a single split
rf = RandomForestClassifier(n_estimators=200, random_state=0)
scores = cross_val_score(rf, X, y, cv=5, scoring="accuracy")
print("CV accuracy per fold:", scores.round(3))
print(f"Mean = {scores.mean():.3f} +/- {scores.std():.3f}")

# 3) Hyper-parameter tuning with GridSearchCV
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    param_grid={"max_depth": [2, 3, 4, 5, 6, None],
                                "min_samples_leaf": [1, 5, 10]},
                    cv=5, scoring="accuracy")
grid.fit(X_train, y_train)
print("\nBest params:", grid.best_params_)
print(f"Best CV accuracy: {grid.best_score_:.3f} | Test accuracy: {grid.score(X_test, y_test):.3f}")

## 4.7 Unsupervised learning: K-Means clustering

K-Means groups data into K clusters:

1. Pick K starting centres. 2. Assign every point to its nearest centre (Euclidean distance). 3. Move each centre to the average of its points. 4. Repeat until the centres stop moving.

Always scale features first, because K-Means uses distances. Choose K with the **elbow method** (where inertia stops dropping sharply), the **silhouette score** (higher is better), and business sense.

> **Real-world example**
>
> RFM segmentation is one of the most used analyses in retail. Customers are described by Recency (days since last order), Frequency (orders per year) and Monetary value (spend). Clusters such as "loyal high spenders", "regulars", "new customers" and "lapsed customers" receive different marketing campaigns. Notice in the output that the silhouette score prefers K = 3, but the elbow and the business need for four campaign groups support K = 4. Metrics guide the choice; the business decides.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# Real-world example: RFM customer segmentation
# Recency = days since last order, Frequency = orders per year, Monetary = yearly spend (Rs)
rng = np.random.default_rng(7)
groups = [  # (n, recency, frequency, monetary)
    (300, 10, 25, 60000),    # loyal high spenders
    (500, 40, 8, 15000),     # regular customers
    (400, 200, 2, 3000),     # lapsed / at-risk customers
    (200, 15, 4, 4000),      # new customers
]
parts = []
for n, r, f, m in groups:
    parts.append(pd.DataFrame({
        "recency": rng.normal(r, r * 0.2, n).clip(1),
        "frequency": rng.normal(f, f * 0.2, n).clip(1),
        "monetary": rng.normal(m, m * 0.2, n).clip(200),
    }))
rfm = pd.concat(parts, ignore_index=True)

X = StandardScaler().fit_transform(rfm)       # K-Means uses distances -> always scale

# Elbow + silhouette to choose K
inertias, sils = [], []
K = range(2, 9)
for k in K:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(K, inertias, "o-"); ax[0].set_title("Elbow method"); ax[0].set_xlabel("K")
ax[1].plot(K, sils, "o-", color="green"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("K")
plt.show()
print("Best K by silhouette:", list(K)[int(np.argmax(sils))])

km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)
rfm["segment"] = km.labels_
profile = rfm.groupby("segment").agg(customers=("recency", "size"),
                                     recency=("recency", "mean"),
                                     frequency=("frequency", "mean"),
                                     monetary=("monetary", "mean")).round(0)
print(profile.sort_values("monetary", ascending=False))

## 4.8 Module 4 practical: handwritten digit recognition

Logistics companies read handwritten PIN codes on parcels automatically. We train and compare four classifiers on 1,797 real 8 x 8 digit images, inspect the confusion matrix, and look at the images the best model gets wrong. This sets up Module 5, where a neural network will solve the harder 28 x 28 version.

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

# PRACTICAL: recognise handwritten digits (think: reading PIN codes on shipping labels)
digits = load_digits()                       # 1,797 real images of 8x8 pixels
print("Data shape:", digits.data.shape, "-> each image is flattened to 64 numbers")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, digits.images, digits.target):
    ax.imshow(img, cmap="gray_r"); ax.set_title(label); ax.axis("off")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42, stratify=digits.target)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN (k=3)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
    "SVM (RBF)": make_pipeline(StandardScaler(), SVC()),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = accuracy_score(y_test, model.predict(X_test))
print(pd.Series(results).sort_values(ascending=False).round(4))

best_name = max(results, key=results.get)
best = models[best_name]
pred = best.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, pred, cmap="Blues")
plt.title(f"{best_name} - confusion matrix")
plt.show()

wrong = np.where(pred != y_test)[0]
print(f"{best_name} misclassified {len(wrong)} of {len(y_test)} test images")
fig, axes = plt.subplots(1, min(6, len(wrong)), figsize=(10, 2), squeeze=False)
for ax, i in zip(axes[0], wrong[:6]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[i]} / pred {pred[i]}", fontsize=8); ax.axis("off")
plt.show()

# Module 5: Deep Learning (35 min)

Deep Learning (DL) is machine learning with neural networks that have many layers. Classical ML works best on tables with hand-made features. Deep learning learns the features itself from raw data, which is why it dominates images, audio and text, and why every LLM is a deep neural network.

## 5.1 The artificial neuron

A neuron multiplies each input by a weight, adds them up with a bias, and passes the result through an activation function:

**z = w1x1 + w2x2 + ... + wnxn + b = w . x + b, output = activation(z)**

The weights say how important each input is, and the bias shifts the decision boundary. Learning means finding good weights and biases. A single neuron with a sigmoid activation is exactly logistic regression.

![Figure 9: A single neuron predicting the probability that a shopper clicks Buy.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d09_neuron.png)
*Figure 9: A single neuron predicting the probability that a shopper clicks Buy.*

In [ ]:
import numpy as np

# A single artificial neuron: z = w . x + b, output = activation(z)
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Real-world example: will this customer click "Buy"?
# features x = [items in cart, minutes on page, has coupon (0/1)]
x = np.array([3, 7.5, 1])
w = np.array([0.4, 0.15, 1.2])     # learned importance of each feature
b = -2.5                            # learned bias (baseline tendency)

z = np.dot(w, x) + b
p = sigmoid(z)
print(f"z = w.x + b = {z:.3f}")
print(f"P(buy) = sigmoid(z) = {p:.3f} -> {'BUY' if p >= 0.5 else 'NO BUY'}")

# The same neuron applied to many customers at once (matrix form)
X = np.array([[3, 7.5, 1],
              [0, 1.0, 0],
              [1, 3.0, 1],
              [5, 12.0, 0]])
probs = sigmoid(X @ w + b)
for row, pr in zip(X, probs):
    print(f"cart={row[0]:.0f}, minutes={row[1]:>4}, coupon={row[2]:.0f} -> P(buy)={pr:.2f}")

## 5.2 Activation functions and softmax

Without a non-linear activation, stacking layers would still produce a straight-line model. Activations let networks learn curves and complex patterns.

| Activation | Output range | Where it is used |
|---|---|---|
| Sigmoid | 0 to 1 | Output layer for yes/no (binary) problems |
| Tanh | -1 to 1 | Hidden layers of older networks, RNNs |
| ReLU = max(0, z) | 0 to infinity | Default for hidden layers: simple and fast to train |
| Leaky ReLU / GELU | Small negative values allowed | GELU is used inside Transformers such as GPT and BERT |
| Softmax | Probabilities that sum to 1 | Output layer for multi-class problems; next-word prediction in LLMs |


> **Real-world example**
>
> When an LLM writes, its last layer produces a score (logit) for every word in its vocabulary, and softmax turns these into probabilities. The **temperature** setting in every LLM API divides the logits before softmax: low temperature makes the top word dominate (focused answers), high temperature flattens the probabilities (creative answers).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-6, 6, 200)
activations = {
    "Sigmoid (0 to 1)": 1 / (1 + np.exp(-z)),
    "Tanh (-1 to 1)": np.tanh(z),
    "ReLU max(0, z)": np.maximum(0, z),
    "Leaky ReLU": np.where(z > 0, z, 0.1 * z),
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, values) in zip(axes, activations.items()):
    ax.plot(z, values, linewidth=2)
    ax.axhline(0, color="grey", linewidth=0.5); ax.axvline(0, color="grey", linewidth=0.5)
    ax.set_title(name)
plt.tight_layout()
plt.show()

# Softmax: turns raw scores (logits) into probabilities that sum to 1.
# This is exactly how an LLM picks the next word from its vocabulary.
def softmax(logits, temperature=1.0):
    scaled = np.array(logits) / temperature
    e = np.exp(scaled - scaled.max())          # subtract max for numerical stability
    return e / e.sum()

words = ["delivered", "late", "broken", "excellent"]
logits = [2.0, 1.0, 0.1, 1.5]
for t in [0.5, 1.0, 2.0]:
    probs = softmax(logits, temperature=t)
    print(f"temperature={t}: " + ", ".join(f"{w}={p:.2f}" for w, p in zip(words, probs)))
print("Low temperature = confident/deterministic, high temperature = more random/creative")

## 5.3 Loss functions and gradient descent

- A **loss function** measures how wrong the predictions are: **Mean Squared Error** for regression, **binary cross-entropy** for yes/no, **categorical cross-entropy** for multi-class (and for LLM training).
- **Gradient descent** reduces the loss step by step: compute the gradient (slope) of the loss for each weight and move the weight a little in the opposite direction: **w = w - learning_rate x gradient**.
- The **learning rate** controls the step size. Too small: very slow. Too large: the loss explodes.
- **Epoch** = one pass over the whole training set. **Batch** = a small group of samples used for one update (mini-batch gradient descent).
- **Adam** is the most used optimiser: gradient descent with momentum and an adaptive learning rate per weight.

![Figure 10: Gradient descent walks downhill on the loss surface (left); the learning rate decides whether it is slow, fast or unstable (right).](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d11_gradient.png)
*Figure 10: Gradient descent walks downhill on the loss surface (left); the learning rate decides whether it is slow, fast or unstable (right).*


> **Real-world example**
>
> A marketing team wants to know how revenue responds to ad spend. We learn the line revenue = w x spend + b from scratch with gradient descent and see how the learning rate changes the result. The same loop, at a much larger scale, trains every neural network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Real-world example: learn  revenue = w * ad_spend + b  with gradient descent
rng = np.random.default_rng(0)
ad_spend = rng.uniform(0, 10, 100)                  # in Rs lakh
revenue = 3.0 * ad_spend + 5 + rng.normal(0, 2, 100)

def train(lr, epochs=1000):
    w, b = 0.0, 0.0
    losses = []
    n = len(ad_spend)
    for _ in range(epochs):
        pred = w * ad_spend + b
        error = pred - revenue
        loss = (error ** 2).mean()                  # Mean Squared Error
        dw = 2 / n * (error * ad_spend).sum()       # dLoss/dw
        db = 2 / n * error.sum()                    # dLoss/db
        w -= lr * dw                                # step downhill
        b -= lr * db
        losses.append(loss)
        if loss > 1e8:                              # stop if training blows up
            break
    return w, b, losses

for lr in [0.001, 0.005, 0.02, 0.04]:
    w, b, losses = train(lr)
    if losses[-1] > 1e8:
        print(f"lr={lr:<6} -> DIVERGED: learning rate too high, the loss exploded")
    else:
        print(f"lr={lr:<6} -> w={w:.2f}, b={b:.2f}, final loss={losses[-1]:.2f}")
    plt.plot(losses, label=f"lr={lr}")

print("True values: w=3.00, b=5.00")
plt.yscale("log"); plt.ylim(1, 1e4); plt.xlabel("Epoch"); plt.ylabel("MSE loss (log scale)")
plt.title("Gradient descent: effect of learning rate"); plt.legend()
plt.show()

## 5.4 Multi-layer networks and backpropagation

A neural network stacks layers: an **input layer** (features), one or more **hidden layers**, and an **output layer**. Training repeats four steps:

1. **Forward pass**: compute predictions layer by layer (Z = XW + b, then activation).
2. **Loss**: compare predictions with the true labels.
3. **Backpropagation**: use the chain rule of calculus to compute how much every weight contributed to the error, moving from the output back to the input.
4. **Update**: change every weight with gradient descent.

![Figure 11: A multi-layer neural network with the forward pass and backpropagation.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d10_network.png)
*Figure 11: A multi-layer neural network with the forward pass and backpropagation.*


The XOR problem below (output 1 only when the two inputs differ) cannot be solved by a single neuron because no straight line separates the classes. A small hidden layer solves it. Here forward pass and backpropagation are written in pure NumPy so every step is visible.

In [ ]:
import numpy as np

# XOR cannot be solved by one neuron (not linearly separable) but a tiny
# 2-layer network can. This is forward pass + backpropagation in pure NumPy.
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])

rng = np.random.default_rng(42)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros((1, 4))     # input(2) -> hidden(4)
W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros((1, 1))     # hidden(4) -> output(1)
sigmoid = lambda z: 1 / (1 + np.exp(-z))
lr = 0.5

for epoch in range(10001):
    # Forward pass
    h = np.tanh(X @ W1 + b1)
    out = sigmoid(h @ W2 + b2)
    loss = -np.mean(y * np.log(out + 1e-9) + (1 - y) * np.log(1 - out + 1e-9))

    # Backward pass (chain rule)
    d_out = (out - y) / len(X)                 # gradient of BCE + sigmoid
    dW2 = h.T @ d_out
    db2 = d_out.sum(axis=0, keepdims=True)
    d_h = (d_out @ W2.T) * (1 - h ** 2)        # derivative of tanh
    dW1 = X.T @ d_h
    db1 = d_h.sum(axis=0, keepdims=True)

    # Gradient descent update
    W2 -= lr * dW2; b2 -= lr * db2
    W1 -= lr * dW1; b1 -= lr * db1

    if epoch % 2000 == 0:
        print(f"epoch {epoch:5d}  loss {loss:.4f}")

print("\nPredictions after training:")
for inp, p in zip(X, out.ravel()):
    print(f"  {inp} -> {p:.3f} -> class {int(p > 0.5)}")

## 5.5 Building networks with Keras

Keras (inside TensorFlow) builds the same networks in a few lines:

- `keras.Sequential([...])` stacks layers; `Dense` is a fully connected layer.
- `Dropout(0.2)` randomly switches off 20% of neurons during training to reduce overfitting.
- `compile()` chooses the optimiser, loss and metrics; `fit()` trains; `evaluate()` tests; `predict()` gives probabilities.
- `EarlyStopping` stops training when the validation loss stops improving and restores the best weights.

> **Real-world example**
>
> MNIST contains 70,000 real handwritten digits (28 x 28 pixels). It is the "hello world" of deep learning and the basis of systems that read cheques, forms and postal codes. A simple network reaches about 98% accuracy in under a minute on Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)

# Real dataset: 70,000 handwritten digits, 28x28 pixels
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0          # scale pixels to 0-1
print("Train:", x_train.shape, "Test:", x_test.shape)

model = keras.Sequential([
    keras.Input(shape=(28, 28)),
    layers.Flatten(),                         # 28x28 -> 784 vector
    layers.Dense(128, activation="relu"),     # hidden layer 1
    layers.Dropout(0.2),                      # regularisation: randomly switch off 20% of neurons
    layers.Dense(64, activation="relu"),      # hidden layer 2
    layers.Dense(10, activation="softmax"),   # 10 classes -> probabilities
])
model.summary()

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=10, batch_size=128,
                    validation_split=0.1, callbacks=[early_stop], verbose=2)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nTest accuracy: {test_acc:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"], label="train")
ax[0].plot(history.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train")
ax[1].plot(history.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.show()

# Predict 5 test images
probs = model.predict(x_test[:5], verbose=0)
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
for ax, img, pr, true in zip(axes, x_test[:5], probs, y_test[:5]):
    ax.imshow(img, cmap="gray_r"); ax.axis("off")
    ax.set_title(f"pred {pr.argmax()} ({pr.max():.0%})\ntrue {true}", fontsize=9)
plt.show()

## 5.6 Convolutional Neural Networks (CNNs)

A dense layer treats an image as a flat list of pixels and ignores which pixels are neighbours. A **convolutional layer** slides small filters (for example 3 x 3) across the image to detect local patterns: edges first, then strokes and shapes in deeper layers. **Max pooling** shrinks the image while keeping the strongest signals. CNNs need far fewer parameters and are more accurate on images.

> **Real-world example**
>
> E-commerce companies use CNNs for visual search ("find similar products from a photo"), detecting damaged items in return photos, and checking that product images meet listing guidelines.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train / 255.0)[..., None]          # add channel axis -> (60000, 28, 28, 1)
x_test = (x_test / 255.0)[..., None]

cnn = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),   # 32 filters learn edges/strokes
    layers.MaxPooling2D(),                                 # downsample 2x
    layers.Conv2D(64, kernel_size=3, activation="relu"),   # 64 filters learn shapes
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
cnn.summary()
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.fit(x_train, y_train, epochs=3, batch_size=128, validation_split=0.1, verbose=2)
print(f"\nCNN test accuracy: {cnn.evaluate(x_test, y_test, verbose=0)[1]:.4f}")
# Tip: in Colab use Runtime > Change runtime type > T4 GPU to train CNNs much faster

## 5.7 Embeddings and attention: the bridge to Transformers and LLMs

- An **embedding layer** is a lookup table that maps each token id to a learnable dense vector. During training, words used in similar contexts get similar vectors, which is the meaning-aware version of the TF-IDF vectors from Module 3.
- **Self-attention** lets every token look at every other token and decide which ones matter for its meaning. In "the phone battery died", the word "died" needs "battery" to be understood.
- Each token embedding is projected into a **Query** (what am I looking for?), a **Key** (what do I contain?) and a **Value** (what information do I pass on?).
- **Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V**: dot products of queries and keys give relevance scores, softmax turns them into weights, and the weights mix the values.
- A **Transformer** stacks many attention + dense layers. GPT, Claude, Gemini and Llama are large Transformers trained to predict the next token.

![Figure 12: Scaled dot-product attention, the core operation of every Transformer.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d12_attention.png)
*Figure 12: Scaled dot-product attention, the core operation of every Transformer.*


Everything in this formula was covered today: dot products and matrix multiplication (Module 3), softmax and probability (Modules 2 and 5), and learned weights (Module 5).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers

# 1) Embedding layer: maps each token id to a learnable dense vector
embedding = layers.Embedding(input_dim=1000, output_dim=8)     # vocab 1000, 8-D vectors
token_ids = np.array([[12, 45, 7, 3]])                          # one sentence of 4 tokens
vectors = embedding(token_ids).numpy()
print("Token ids shape:", token_ids.shape, "-> embeddings shape:", vectors.shape)

# 2) Scaled dot-product attention - the core operation inside every Transformer/LLM
#    Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

tokens = ["the", "phone", "battery", "died"]
rng = np.random.default_rng(1)
d_model = 6
X = rng.normal(0, 1, (4, d_model))                 # pretend these are the 4 token embeddings
X[2] = X[1] * 0.9 + rng.normal(0, 0.2, d_model)    # make "battery" related to "phone"
X[3] = X[2] * 0.8 + rng.normal(0, 0.3, d_model)    # and "died" related to "battery"

Wq, Wk, Wv = (rng.normal(0, 0.5, (d_model, d_model)) for _ in range(3))
Q, K, V = X @ Wq, X @ Wk, X @ Wv                   # queries, keys, values

scores = Q @ K.T / np.sqrt(d_model)                # how much each token "looks at" others
weights = softmax(scores)                          # each row sums to 1
output = weights @ V                               # context-aware token representations

print("\nAttention weights (rows = query token):")
print("         " + "".join(f"{t:>9}" for t in tokens))
for t, row in zip(tokens, weights):
    print(f"{t:>8} " + "".join(f"{w:9.2f}" for w in row))
print("Output shape:", output.shape, "(same as input: 4 tokens x 6 dims)")

plt.imshow(weights, cmap="Blues")
plt.xticks(range(4), tokens); plt.yticks(range(4), tokens)
plt.colorbar(); plt.title("Self-attention weights")
plt.show()

## 5.8 Module 5 practical: review sentiment classifier

We train a neural network on 50,000 real IMDB movie reviews to classify sentiment, then score our own sentences. The model learns a 32-dimensional embedding for every word and averages them to represent a review. The same architecture can classify product reviews, support tickets or chat messages.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# PRACTICAL: review sentiment classifier on 50,000 real IMDB movie reviews
tf.random.set_seed(42)
VOCAB, MAXLEN = 10_000, 200

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB)
print("Reviews:", len(x_train), "train /", len(x_test), "test")

# Every review is already a list of word ids; pad/cut them to the same length
x_train = keras.utils.pad_sequences(x_train, maxlen=MAXLEN)
x_test = keras.utils.pad_sequences(x_test, maxlen=MAXLEN)

model = keras.Sequential([
    keras.Input(shape=(MAXLEN,)),
    layers.Embedding(VOCAB, 32),                 # each word -> 32-D learned vector
    layers.GlobalAveragePooling1D(),             # average word vectors -> review vector
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),       # P(positive)
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=256, validation_split=0.2, verbose=2)
print(f"\nTest accuracy: {model.evaluate(x_test, y_test, verbose=0)[1]:.4f}")

# Score our own reviews: convert words to the same ids the dataset uses
word_index = keras.datasets.imdb.get_word_index()
def encode(text):
    ids = [1]                                    # 1 = start token
    for word in text.lower().replace(",", " ").replace(".", " ").split():
        idx = word_index.get(word)
        ids.append(idx + 3 if idx is not None and idx + 3 < VOCAB else 2)   # 2 = unknown word
    return keras.utils.pad_sequences([ids], maxlen=MAXLEN)

for review in ["An excellent and wonderful story, brilliant acting",
               "Boring, waste of time. The worst plot and terrible acting",
               "It was okay, some good moments but too long"]:
    p = float(model.predict(encode(review), verbose=0)[0, 0])
    print(f"{p:.2f} {'POSITIVE' if p > 0.5 else 'NEGATIVE'} : {review}")

> **Discussion**
>
> The model reaches about 86% accuracy with a very simple architecture. Fine-tuned Transformers reach 93 to 95% on this task because attention understands word order and negation ("not good"). In GenAI projects, the usual choices are calling an LLM with a prompt, using its embeddings plus a small classifier, or fine-tuning.

# Module 6: End-to-End Project - E-commerce Order Return Prediction (30 min)

Returns cost Indian e-commerce companies a lot in reverse logistics, repackaging and lost sales, especially in fashion. In this project you build a complete solution on a dataset of 10,000 orders, using everything from Modules 1 to 5.

## 6.1 Business problem and dataset

**Goal**: score every order right after delivery (when the delivery time and the customer's rating are known) with a return-risk probability, so the support team can call high-risk customers and offer an exchange, size help or a fix before a return is raised.

**Dataset**: `ecommerce_orders_10k.xlsx`, sheet `Orders`, 10,000 orders from calendar year 2025 across 20 Indian cities and 7 categories. The sheet `Data_Dictionary` explains every column. The data is realistic but simulated, and it deliberately contains missing values for cleaning practice.

| Column | Description |
|---|---|
| Order_ID, Order_Date, Customer_ID | Order identifier, date-time of order, customer identifier |
| Customer_Age, Gender, City, City_Tier | Customer details (Age has some missing values) |
| Category, Product_Name | One of 7 categories and 41 products |
| Unit_Price, Quantity, Discount_Pct, Order_Value | Pricing (Order_Value = price x quantity x (1 - discount)) |
| Payment_Method, Device_Type, Is_Member | UPI, cards, COD, net banking, wallet; app, mobile web or desktop; membership |
| Previous_Orders, Session_Duration_Min | Customer history and browsing time |
| Delivery_Days, Customer_Rating, Review_Text | Post-delivery information (rating and review have missing values) |
| Returned | TARGET: Yes / No (about 18% Yes) |


![Figure 13: Project architecture.](https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/images/d13_project.png)
*Figure 13: Project architecture.*


> **How to run the project in Colab**
>
> 1. Open a new Colab notebook. 2. Run Step 0; a file picker appears; choose `ecommerce_orders_10k.xlsx` (alternatively drag the file into the Files panel on the left before running). 3. Run Steps 1 to 14 in order, each in its own cell. Later steps use variables created by earlier steps. Total run time is about 2 to 3 minutes on the free CPU runtime.

## Step 0: Setup and data upload

In [ ]:
# STEP 0: Imports and data loading
import os
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")

FILE = "ecommerce_orders_10k.xlsx"
DATA_URL = "https://raw.githubusercontent.com/anandjha90/Advanced-Python-for-GenAI-Prep-Training/main/data/ecommerce_orders_10k.xlsx"

# 1) use a local copy if it exists (repo cloned locally or file uploaded to Colab)
for candidate in [FILE, os.path.join("..", "data", FILE), os.path.join("data", FILE)]:
    if os.path.exists(candidate):
        FILE = candidate
        break
else:
    # 2) otherwise download it from the GitHub repo
    try:
        urllib.request.urlretrieve(DATA_URL, FILE)
        print("Downloaded dataset from GitHub")
    except Exception as err:
        # 3) last resort: upload it manually in Colab
        print("Download failed:", err)
        from google.colab import files
        files.upload()
print("Using file:", FILE, "| exists:", os.path.exists(FILE))

## Step 1: Load and understand the data

Check the shape, data types, missing values and the balance of the target. About 18% of orders are returned, so the classes are imbalanced: a model that always says "No" would be 82% accurate and useless. This is why we will focus on recall, precision, F1 and ROC-AUC rather than accuracy.

In [ ]:
# STEP 1: Load and understand the data
df = pd.read_excel(FILE, sheet_name="Orders")
print("Shape:", df.shape)
print(df.head())
print()
df.info()
print("\nMissing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nTarget distribution:\n", df["Returned"].value_counts(normalize=True).round(3))
print("\nNumeric summary:\n", df.describe(include="number").round(1).T)

## Step 2: Clean the data

Missing ages are filled with the median, missing devices become "Unknown", and missing ratings are filled with the median rating of the same category (a smarter rule than the global median). The target is converted to 0/1.

In [ ]:
# STEP 2: Clean the data
df = df.drop_duplicates(subset="Order_ID").copy()
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

# Fill missing values with sensible, explainable rules
df["Customer_Age"] = df["Customer_Age"].fillna(df["Customer_Age"].median())
df["Device_Type"] = df["Device_Type"].fillna("Unknown")
df["Customer_Rating"] = df["Customer_Rating"].fillna(
    df.groupby("Category")["Customer_Rating"].transform("median"))   # median of same category
df["Review_Text"] = df["Review_Text"].fillna("No review")

# Target to 0/1
df["Returned_Flag"] = (df["Returned"] == "Yes").astype(int)

# Sanity checks
assert df["Order_Value"].min() > 0
assert df[["Customer_Age", "Device_Type", "Customer_Rating"]].isna().sum().sum() == 0
print("Rows after cleaning:", len(df))
print("Remaining missing values:", df.isna().sum().sum())
print(f"Overall return rate: {df['Returned_Flag'].mean():.1%}")

## Step 3: Exploratory data analysis

Charts reveal the patterns the model will learn: Fashion has by far the highest return rate, COD orders return more, returns rise with delivery days, and low ratings are a strong signal.

In [ ]:
# STEP 3: Exploratory data analysis (EDA)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

rate = df.groupby("Category")["Returned_Flag"].mean().sort_values(ascending=False) * 100
rate.plot(kind="bar", ax=axes[0, 0], color="indianred")
axes[0, 0].set_title("Return rate by category (%)"); axes[0, 0].tick_params(axis="x", rotation=45)

pay = df.groupby("Payment_Method")["Returned_Flag"].mean().sort_values(ascending=False) * 100
pay.plot(kind="bar", ax=axes[0, 1], color="darkorange")
axes[0, 1].set_title("Return rate by payment method (%)"); axes[0, 1].tick_params(axis="x", rotation=45)

dd = df.groupby("Delivery_Days")["Returned_Flag"].mean() * 100
dd.plot(marker="o", ax=axes[0, 2], color="purple")
axes[0, 2].set_title("Return rate vs delivery days (%)")

rr = df.groupby("Customer_Rating")["Returned_Flag"].mean() * 100
rr.plot(kind="bar", ax=axes[1, 0], color="teal")
axes[1, 0].set_title("Return rate by customer rating (%)")

sns.histplot(np.log10(df["Order_Value"]), bins=50, ax=axes[1, 1], color="steelblue")
axes[1, 1].set_title("Order value (log10 scale)")

num_cols = ["Customer_Age", "Unit_Price", "Quantity", "Discount_Pct", "Order_Value",
            "Previous_Orders", "Session_Duration_Min", "Delivery_Days", "Customer_Rating",
            "Returned_Flag"]
sns.heatmap(df[num_cols].corr(), cmap="coolwarm", center=0, ax=axes[1, 2], annot=False)
axes[1, 2].set_title("Correlation heatmap")
plt.tight_layout()
plt.show()

# Monthly revenue trend
monthly = df.set_index("Order_Date").resample("MS")["Order_Value"].sum() / 1e5
monthly.plot(marker="o", figsize=(12, 4), title="Monthly revenue (Rs lakh)")
plt.show()
city_revenue = df.groupby("City")["Order_Value"].sum() / 1e5
print("Top 5 cities by revenue (Rs lakh):\n", city_revenue.nlargest(5).round(1))

## Step 4: Statistical tests

Before trusting the patterns we test them (Module 2). Chi-square tests confirm that category and payment method are related to returns, a Welch t-test confirms that returned orders had longer deliveries, and conditional probabilities quantify the risk.

In [ ]:
# STEP 4: Statistical tests - are the patterns real or just noise?
# 1) Chi-square: is Category related to Returned?
ct = pd.crosstab(df["Category"], df["Returned_Flag"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Category vs Returned       : chi2={chi2:.1f}, p={p:.2e}")

# 2) Chi-square: is Payment_Method related to Returned?
ct = pd.crosstab(df["Payment_Method"], df["Returned_Flag"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f"Payment method vs Returned : chi2={chi2:.1f}, p={p:.2e}")

# 3) Welch t-test: do returned orders have longer delivery times?
ret = df.loc[df["Returned_Flag"] == 1, "Delivery_Days"]
kept = df.loc[df["Returned_Flag"] == 0, "Delivery_Days"]
t, p = stats.ttest_ind(ret, kept, equal_var=False)
print(f"Delivery days (returned {ret.mean():.2f} vs kept {kept.mean():.2f}): t={t:.2f}, p={p:.2e}")

# 4) Bayes-style conditional probabilities
p_ret = df["Returned_Flag"].mean()
p_ret_cod = df.loc[df["Payment_Method"] == "COD", "Returned_Flag"].mean()
fashion_low = (df["Category"] == "Fashion") & (df["Customer_Rating"] <= 2)
p_ret_fashion_low = df.loc[fashion_low, "Returned_Flag"].mean()
print(f"\nP(return)                         = {p_ret:.3f}")
print(f"P(return | COD)                   = {p_ret_cod:.3f}")
print(f"P(return | Fashion & rating <= 2) = {p_ret_fashion_low:.3f}")

# 5) 95% confidence interval for the overall return rate
n = len(df)
se = np.sqrt(p_ret * (1 - p_ret) / n)
print(f"95% CI for return rate: {p_ret - 1.96 * se:.3f} to {p_ret + 1.96 * se:.3f}")

## Step 5: Feature engineering

New features capture business knowledge: time features (month, weekday, hour, weekend, sale month), flags for high discount, slow delivery, new customer and COD, and a log transform of the skewed order value. Identifier columns and free text are not used as model inputs.

In [ ]:
# STEP 5: Feature engineering
# Business use case: score every order right after delivery + rating, so the support
# team can reach out (exchange, size help, fix) BEFORE the customer raises a return.
df["Order_Month"] = df["Order_Date"].dt.month
df["Order_Weekday"] = df["Order_Date"].dt.dayofweek
df["Order_Hour"] = df["Order_Date"].dt.hour
df["Is_Weekend"] = (df["Order_Weekday"] >= 5).astype(int)
df["Is_Sale_Month"] = df["Order_Month"].isin([1, 8, 10, 11]).astype(int)
df["High_Discount"] = (df["Discount_Pct"] > 40).astype(int)
df["Slow_Delivery"] = (df["Delivery_Days"] > 5).astype(int)
df["Is_New_Customer"] = (df["Previous_Orders"] == 0).astype(int)
df["Log_Order_Value"] = np.log1p(df["Order_Value"])
df["Is_COD"] = (df["Payment_Method"] == "COD").astype(int)

numeric_features = ["Customer_Age", "City_Tier", "Unit_Price", "Quantity", "Discount_Pct",
                    "Log_Order_Value", "Previous_Orders", "Session_Duration_Min", "Delivery_Days",
                    "Customer_Rating", "Order_Month", "Order_Weekday", "Order_Hour", "Is_Weekend",
                    "Is_Sale_Month", "High_Discount", "Slow_Delivery", "Is_New_Customer", "Is_COD"]
categorical_features = ["Gender", "Category", "Payment_Method", "Device_Type", "Is_Member"]
TARGET = "Returned_Flag"

X = df[numeric_features + categorical_features]
y = df[TARGET]
print("Feature matrix:", X.shape, "| positives:", y.sum())

## Step 6: Split and preprocess

A stratified 80/20 split keeps the 18% return rate in both sets. The `ColumnTransformer` scales numeric features and one-hot encodes categorical features, learning only from the training data.

In [ ]:
# STEP 6: Train/test split and preprocessing pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
print("Train:", X_train_p.shape, "| Test:", X_test_p.shape)
print(f"Return rate train {y_train.mean():.3f} | test {y_test.mean():.3f}")

## Step 7: Machine learning models

We train logistic regression, Random Forest and gradient boosting. `class_weight="balanced"` tells each model to pay more attention to the minority class (returns).

In [ ]:
# STEP 7: Train and compare machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced_subsample",
                                            random_state=42, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                        class_weight="balanced", random_state=42),
}

def score_model(name, y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    return {"model": name,
            "accuracy": accuracy_score(y_true, pred),
            "precision": precision_score(y_true, pred),
            "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred),
            "roc_auc": roc_auc_score(y_true, proba)}

results, probas = [], {}
for name, model in models.items():
    model.fit(X_train_p, y_train)
    probas[name] = model.predict_proba(X_test_p)[:, 1]
    results.append(score_model(name, y_test, probas[name]))

results_df = pd.DataFrame(results).set_index("model").round(3)
print(results_df)

best_ml = results_df["roc_auc"].idxmax()
print(f"\nBest ML model by ROC-AUC: {best_ml}")
best_pred = (probas[best_ml] >= 0.5).astype(int)
print(classification_report(y_test, best_pred, target_names=["Kept", "Returned"]))

## Step 8: Business-driven threshold

A probability of 0.5 is not automatically the right cut-off. We assume a call costs Rs 30 and a prevented return saves Rs 250, with 40% of contacted risky customers saved. The table shows the trade-off between catching more returns and making more calls, and we pick the threshold with the highest net saving.

In [ ]:
# STEP 8: Choose a decision threshold that suits the business
# Calling a customer costs ~Rs 30; a prevented return saves ~Rs 250 of reverse logistics.
from sklearn.metrics import confusion_matrix

proba = probas[best_ml]
rows = []
for th in np.arange(0.30, 0.81, 0.05):
    tn, fp, fn, tp = confusion_matrix(y_test, (proba >= th).astype(int)).ravel()
    saving = tp * 250 * 0.4 - (tp + fp) * 30      # assume 40% of contacted risky orders are saved
    rows.append({"threshold": round(th, 2), "flagged": tp + fp, "caught_returns": tp,
                 "missed_returns": fn, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn),
                 "net_saving_Rs": saving})
th_df = pd.DataFrame(rows).round(3)
print(th_df.to_string(index=False))
best_th = th_df.loc[th_df["net_saving_Rs"].idxmax(), "threshold"]
print(f"\nBest threshold for net saving: {best_th}")

## Step 9: Deep learning model

A Keras network with two hidden layers, batch normalisation and dropout is trained on the same preprocessed features, with class weights and early stopping on validation AUC.

In [ ]:
# STEP 9: Deep learning model (Keras neural network)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

n_features = X_train_p.shape[1]
nn = keras.Sequential([
    keras.Input(shape=(n_features,)),
    layers.Dense(64, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
nn.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
           loss="binary_crossentropy",
           metrics=[keras.metrics.AUC(name="auc")])

# Returns are the minority class -> give them more weight
neg, pos = np.bincount(y_train)
class_weight = {0: 1.0, 1: neg / pos}

early_stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5,
                                           restore_best_weights=True)
history = nn.fit(X_train_p, y_train.values, validation_split=0.2, epochs=60, batch_size=64,
                 class_weight=class_weight, callbacks=[early_stop], verbose=0)
print("Epochs trained:", len(history.history["loss"]))

probas["Neural Network"] = nn.predict(X_test_p, verbose=0).ravel()
nn_scores = score_model("Neural Network", y_test, probas["Neural Network"])
nn_scores.pop("model")
results_df.loc["Neural Network"] = pd.Series(nn_scores).round(3)
print(results_df)

plt.figure(figsize=(8, 4))
plt.plot(history.history["auc"], label="train AUC")
plt.plot(history.history["val_auc"], label="validation AUC")
plt.xlabel("Epoch"); plt.title("Neural network training"); plt.legend()
plt.show()

## Step 10: Compare all models

In [ ]:
# STEP 10: Compare all models with ROC curves and confusion matrices
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, pr in probas.items():
    fpr, tpr, _ = roc_curve(y_test, pr)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, pr):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate")
axes[0].set_title("ROC curves"); axes[0].legend()

ConfusionMatrixDisplay.from_predictions(y_test, (probas[best_ml] >= 0.5).astype(int),
                                        display_labels=["Kept", "Returned"], cmap="Blues", ax=axes[1])
axes[1].set_title(f"{best_ml} - confusion matrix")
plt.tight_layout()
plt.show()
print(results_df.sort_values("roc_auc", ascending=False))

> **Interpretation**
>
> All models reach a ROC-AUC of about 0.72 to 0.77. Logistic regression is the best here and catches about 70% of returns, with the other models close behind. On structured business tables with around 10,000 rows, simple models and gradient-boosted trees often match or beat neural networks; deep learning shines on images, audio and text. Exact numbers can vary slightly between runs and library versions.

## Step 11: Feature importance and business insights

In [ ]:
# STEP 11: What drives returns? Feature importance + business insights
rf = models["Random Forest"]
importance = (pd.Series(rf.feature_importances_, index=feature_names)
                .sort_values(ascending=False).head(15))
importance.index = importance.index.str.replace("num__", "").str.replace("cat__", "")
importance.sort_values().plot(kind="barh", figsize=(9, 6), color="seagreen",
                              title="Top 15 drivers of returns (Random Forest)")
plt.show()

print("BUSINESS INSIGHTS")
fashion = df.loc[df.Category == "Fashion", "Returned_Flag"].mean()
grocery = df.loc[df.Category == "Grocery", "Returned_Flag"].mean()
print(f"1. Fashion return rate {fashion:.0%} vs Grocery {grocery:.0%} -> add size guides / fit tools.")
print(f"2. COD orders return {p_ret_cod:.0%} vs overall {p_ret:.0%} -> nudge UPI with prepaid offers.")
slow = df.loc[df.Slow_Delivery == 1, 'Returned_Flag'].mean()
fast = df.loc[df.Slow_Delivery == 0, 'Returned_Flag'].mean()
print(f"3. Slow deliveries (>5 days) return {slow:.0%} vs {fast:.0%} -> prioritise tier-3 logistics.")
low = df.loc[df.Customer_Rating <= 2, 'Returned_Flag'].mean()
print(f"4. Ratings of 1-2 stars return {low:.0%} -> trigger instant support call on low ratings.")

## Step 12: Save the model and score new orders

The full pipeline (preprocessing + model) is saved with `joblib` so it can be loaded in an API or a batch job. A helper function rebuilds the engineered features for new orders, and each order gets a risk score and an action.

In [ ]:
# STEP 12: Save the best pipeline and score new orders
import joblib
from sklearn.pipeline import Pipeline

final_model = Pipeline([("prep", preprocessor), ("model", models[best_ml])])
final_model.fit(X_train, y_train)                 # refit the whole pipeline on raw features
joblib.dump(final_model, "return_risk_model.joblib")
print("Model saved: return_risk_model.joblib")

def add_features(orders):
    o = orders.copy()
    o["Order_Date"] = pd.to_datetime(o["Order_Date"])
    o["Order_Month"] = o["Order_Date"].dt.month
    o["Order_Weekday"] = o["Order_Date"].dt.dayofweek
    o["Order_Hour"] = o["Order_Date"].dt.hour
    o["Is_Weekend"] = (o["Order_Weekday"] >= 5).astype(int)
    o["Is_Sale_Month"] = o["Order_Month"].isin([1, 8, 10, 11]).astype(int)
    o["High_Discount"] = (o["Discount_Pct"] > 40).astype(int)
    o["Slow_Delivery"] = (o["Delivery_Days"] > 5).astype(int)
    o["Is_New_Customer"] = (o["Previous_Orders"] == 0).astype(int)
    o["Order_Value"] = o["Unit_Price"] * o["Quantity"] * (1 - o["Discount_Pct"] / 100)
    o["Log_Order_Value"] = np.log1p(o["Order_Value"])
    o["Is_COD"] = (o["Payment_Method"] == "COD").astype(int)
    return o[numeric_features + categorical_features]

new_orders = pd.DataFrame([
    {"Order_Date": "2026-01-10 21:15", "Customer_Age": 22, "Gender": "Female", "City_Tier": 3,
     "Category": "Fashion", "Unit_Price": 1499, "Quantity": 2, "Discount_Pct": 45,
     "Payment_Method": "COD", "Device_Type": "Mobile App", "Is_Member": "No", "Previous_Orders": 0,
     "Session_Duration_Min": 6.5, "Delivery_Days": 8, "Customer_Rating": 2},
    {"Order_Date": "2026-01-12 10:05", "Customer_Age": 38, "Gender": "Male", "City_Tier": 1,
     "Category": "Books", "Unit_Price": 549, "Quantity": 1, "Discount_Pct": 10,
     "Payment_Method": "UPI", "Device_Type": "Desktop", "Is_Member": "Yes", "Previous_Orders": 14,
     "Session_Duration_Min": 18.0, "Delivery_Days": 2, "Customer_Rating": 5},
])
loaded = joblib.load("return_risk_model.joblib")
risk = loaded.predict_proba(add_features(new_orders))[:, 1]
for (_, o), r in zip(new_orders.iterrows(), risk):
    action = "CALL CUSTOMER" if r >= best_th else "no action"
    print(f"{o['Category']:8} | {o['Payment_Method']:4} | rating {o['Customer_Rating']} "
          f"-> return risk {r:.0%} -> {action}")

## Step 13 (bonus): Customer segmentation

Customers are aggregated to Recency, Frequency and Monetary values and clustered with K-Means (Module 4). The log transform reduces the effect of a few very large customers.

In [ ]:
# STEP 13 (bonus): Customer segmentation with K-Means (RFM)
from sklearn.cluster import KMeans

snapshot = df["Order_Date"].max() + pd.Timedelta(days=1)
rfm = df.groupby("Customer_ID").agg(
    Recency=("Order_Date", lambda d: (snapshot - d.max()).days),
    Frequency=("Order_ID", "count"),
    Monetary=("Order_Value", "sum"),
    Return_Rate=("Returned_Flag", "mean"))

rfm_scaled = StandardScaler().fit_transform(np.log1p(rfm[["Recency", "Frequency", "Monetary"]]))
kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm["Segment"] = kmeans.fit_predict(rfm_scaled)

profile = rfm.groupby("Segment").agg(Customers=("Recency", "size"), Recency=("Recency", "mean"),
                                     Frequency=("Frequency", "mean"), Monetary=("Monetary", "mean"),
                                     Return_Rate=("Return_Rate", "mean")).round(2)
profile = profile.sort_values("Monetary", ascending=False)
profile["Label"] = ["Champions", "Loyal", "Potential", "At risk / one-time"]
print(profile)

## Step 14 (GenAI bridge): semantic search over reviews

The final step connects the project to Generative AI. Review texts are converted to TF-IDF vectors and searched with cosine similarity (Module 3), which is the retrieval half of a RAG system. The retrieved reviews are then placed into a prompt that can be sent to any LLM to explain why customers return products. In the GenAI program, TF-IDF will be replaced by neural embeddings and a vector database.

In [ ]:
# STEP 14 (GenAI bridge): semantic search over customer reviews
# This is the "retrieval" half of a RAG chatbot that answers: "Why are customers returning X?"
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cols = ["Order_ID", "Category", "Review_Text", "Returned"]
reviews = df.loc[df["Review_Text"] != "No review", cols].reset_index(drop=True)
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
review_vectors = vectorizer.fit_transform(reviews["Review_Text"])
print("Review vectors:", review_vectors.shape)

def find_reviews(query, top_k=5, category=None):
    scores = cosine_similarity(vectorizer.transform([query]), review_vectors).ravel()
    result = reviews.assign(score=scores)
    if category:
        result = result[result["Category"] == category]
    result = result.sort_values("score", ascending=False).drop_duplicates("Review_Text")
    return result.head(top_k)

print("\nQuery: 'size did not fit' in Fashion")
show = ["Order_ID", "Review_Text", "score"]
print(find_reviews("size did not fit", category="Fashion")[show].to_string(index=False))

print("\nQuery: 'damaged product not working' in Electronics")
print(find_reviews("damaged product not working", category="Electronics")[show].to_string(index=False))

# Build an LLM-ready prompt from returned-order reviews (send it to any LLM API)
sample = (reviews[(reviews["Category"] == "Fashion") & (reviews["Returned"] == "Yes")]["Review_Text"]
          .drop_duplicates().sample(10, random_state=1))
context = "\n".join("- " + t for t in sample)
prompt = ("You are an e-commerce analyst. Using only the reviews below, list the top reasons "
          f"customers return Fashion products and suggest 3 fixes.\n\nReviews:\n{context}")
print("\nPrompt for the LLM:\n", prompt)

## 6.2 Project summary and extensions

| Module | Where it was used in the project |
|---|---|
| Python | Pandas cleaning, functions for feature engineering, dictionaries of models, joblib, f-string prompts |
| Statistics | Missing-value strategy, chi-square and t-tests, conditional probabilities, confidence interval |
| Linear Algebra | Feature matrix, scaling, one-hot vectors, cosine similarity for review search |
| Machine Learning | Pipeline, logistic regression, Random Forest, gradient boosting, metrics, threshold, K-Means |
| Deep Learning | Keras network with dropout, batch normalisation, class weights and early stopping |
| GenAI | Retrieval over reviews and an LLM-ready prompt |


**Try these extensions**: add `Product_Name` as a feature and check whether AUC improves; tune the Random Forest with `GridSearchCV`; replace TF-IDF with a sentence-embedding model (`pip install sentence-transformers`) and compare search results; send the Step 14 prompt to an LLM API and review its answer.